# Cell 1 — Load the Pangenome Feature Engineering Cohort

In [ ]:
from google.colab import drive

import os
import pandas as pd


# =========================================================
# 1. Mount Google Drive
# =========================================================

drive.mount(
    "/content/drive"
)


# =========================================================
# 2. Project paths
# =========================================================

DRIVE_ROOT = (
    "/content/drive/MyDrive/AMR-Genome-ML"
)

INPUT_MANIFEST_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "processed",
    "genomic_feature_input_manifest.csv"
)

FILE04_SUMMARY_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "file04_known_amr_feature_engineering_summary.csv"
)

X_KNOWN_AMR_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "X_known_amr_final.csv"
)


# =========================================================
# 3. File 05 directories
# =========================================================

PANGENOME_DIR = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "pangenome"
)

PANGENOME_CHECKPOINT_DIR = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "pangenome"
)

PANGENOME_TMP_DIR = (
    "/content/pangenome_work"
)


for path in [
    PANGENOME_DIR,
    PANGENOME_CHECKPOINT_DIR,
    PANGENOME_TMP_DIR
]:

    os.makedirs(
        path,
        exist_ok=True
    )


# =========================================================
# 4. Verify required upstream outputs
# =========================================================

required_files = [
    INPUT_MANIFEST_PATH,
    FILE04_SUMMARY_PATH,
    X_KNOWN_AMR_PATH
]


missing_files = [
    path
    for path in required_files
    if not os.path.exists(path)
]


if missing_files:

    raise FileNotFoundError(
        "Missing required upstream files:\n"
        + "\n".join(
            missing_files
        )
    )


# =========================================================
# 5. Load finalized study cohort
# =========================================================

df_pangenome_input = pd.read_csv(
    INPUT_MANIFEST_PATH,
    dtype={
        "Genome ID": str,
        "Taxon ID": str
    }
)


df_pangenome_input[
    "Genome ID"
] = (
    df_pangenome_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


# =========================================================
# 6. Cohort integrity
# =========================================================

assert len(
    df_pangenome_input
) == 4233, (
    f"Expected 4,233 study genomes, "
    f"found {len(df_pangenome_input):,}."
)


assert (
    df_pangenome_input[
        "Genome ID"
    ]
    .nunique()
    == 4233
), "Duplicate Genome IDs detected."


assert (
    df_pangenome_input[
        "Complete FASTA"
    ]
    .eq(True)
    .all()
), "Incomplete FASTA status detected."


# =========================================================
# 7. Verify every FASTA file still exists
# =========================================================

df_pangenome_input[
    "FASTA Exists"
] = (
    df_pangenome_input[
        "FASTA Path"
    ]
    .astype(str)
    .apply(
        os.path.exists
    )
)


missing_fasta = (
    df_pangenome_input[
        ~df_pangenome_input[
            "FASTA Exists"
        ]
    ]
    .copy()
)


assert missing_fasta.empty, (
    f"{len(missing_fasta)} FASTA files "
    f"are missing from persistent storage."
)


# =========================================================
# 8. Verify File 04 final output for continuity
# =========================================================

df_file04_summary = pd.read_csv(
    FILE04_SUMMARY_PATH
)


df_x_known_amr_header = pd.read_csv(
    X_KNOWN_AMR_PATH,
    nrows=1
)


known_amr_features = (
    len(
        df_x_known_amr_header.columns
    )
    - 1
)


assert known_amr_features == 668, (
    f"Expected 668 finalized known-AMR features, "
    f"found {known_amr_features:,}."
)


# =========================================================
# 9. Phenotype distribution
# =========================================================

phenotype_counts = (
    df_pangenome_input[
        "Resistant Phenotype"
    ]
    .value_counts()
)


# =========================================================
# 10. File 05 input summary
# =========================================================

print("=" * 70)
print("FILE 05 — PANGENOME FEATURE ENGINEERING")
print("=" * 70)

print(
    f"Study genomes             : "
    f"{len(df_pangenome_input):,}"
)

print(
    f"Unique Genome IDs         : "
    f"{df_pangenome_input['Genome ID'].nunique():,}"
)

print(
    f"Available FASTA files     : "
    f"{df_pangenome_input['FASTA Exists'].sum():,}"
)

print(
    f"Known-AMR baseline        : "
    f"{known_amr_features:,} features"
)

print()

print(
    "Phenotype distribution:"
)

print(
    phenotype_counts
)

print()

print(
    "Pangenome output directory:"
)

print(
    PANGENOME_DIR
)

print()

print(
    "PASS: File 05 input cohort is complete "
    "and ready for pangenome strategy benchmarking."
)

Mounted at /content/drive
FILE 05 — PANGENOME FEATURE ENGINEERING
Study genomes             : 4,233
Unique Genome IDs         : 4,233
Available FASTA files     : 4,233
Known-AMR baseline        : 668 features

Phenotype distribution:
Resistant Phenotype
Susceptible    2630
Resistant      1603
Name: count, dtype: int64

Pangenome output directory:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome

PASS: File 05 input cohort is complete and ready for pangenome strategy benchmarking.


# Cell 2 — Audit Precomputed BV-BRC Protein-Family Coverage

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd


# =========================================================
# 1. BV-BRC genome API
# =========================================================

BV_BRC_GENOME_API = (
    "https://www.bv-brc.org/api/genome/"
)

session = requests.Session()

session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Study Genome IDs
# =========================================================

study_genome_ids = (
    df_pangenome_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)


assert len(study_genome_ids) == 4233


# =========================================================
# 3. Batch configuration
# =========================================================

BATCH_SIZE = 200

batches = [
    study_genome_ids[i:i + BATCH_SIZE]
    for i in range(
        0,
        len(study_genome_ids),
        BATCH_SIZE
    )
]


family_metadata_records = []
batch_audit = []


print("=" * 70)
print("BV-BRC PLFAM COVERAGE RETRIEVAL")
print("=" * 70)

print(
    f"Study genomes : "
    f"{len(study_genome_ids):,}"
)

print(
    f"Total batches : "
    f"{len(batches)}"
)

print()


# =========================================================
# 4. Retrieve PLFam coverage metadata
# =========================================================

for batch_index, batch_ids in enumerate(
    batches,
    start=1
):

    id_string = ",".join(
        batch_ids
    )


    query_url = (
        f"{BV_BRC_GENOME_API}"
        f"?in(genome_id,({id_string}))"
        f"&select("
        f"genome_id,"
        f"patric_cds,"
        f"plfam_cds,"
        f"plfam_cds_ratio"
        f")"
        f"&limit(10000)"
    )


    start_time = time.time()

    success = False
    records = []
    last_error = None


    for attempt in range(
        1,
        4
    ):

        try:

            response = session.get(
                query_url,
                timeout=120
            )

            response.raise_for_status()

            records = response.json()


            if not isinstance(
                records,
                list
            ):

                raise RuntimeError(
                    "Unexpected BV-BRC response format."
                )


            family_metadata_records.extend(
                records
            )

            success = True

            break


        except Exception as exc:

            last_error = str(
                exc
            )

            if attempt < 3:

                time.sleep(
                    2 * attempt
                )


    elapsed = (
        time.time()
        - start_time
    )


    batch_audit.append(
        {
            "Batch":
                batch_index,

            "Requested":
                len(batch_ids),

            "Returned":
                len(records)
                if success
                else 0,

            "Status":
                "PASS"
                if success
                else "FAIL",

            "Runtime Seconds":
                round(
                    elapsed,
                    2
                ),

            "Error":
                None
                if success
                else last_error
        }
    )


    print(
        f"[{batch_index:02d}/{len(batches)}] "
        f"requested={len(batch_ids):3d} | "
        f"returned={len(records) if success else 0:3d} | "
        f"{'PASS' if success else 'FAIL'}"
    )


# =========================================================
# 5. Verify retrieval
# =========================================================

df_plfam_batch_audit = pd.DataFrame(
    batch_audit
)


failed_batches = (
    df_plfam_batch_audit[
        df_plfam_batch_audit[
            "Status"
        ]
        .ne("PASS")
    ]
)


if not failed_batches.empty:

    display(
        failed_batches
    )

    raise RuntimeError(
        "One or more BV-BRC batches failed."
    )


df_plfam_metadata = pd.DataFrame(
    family_metadata_records
)


df_plfam_metadata[
    "genome_id"
] = (
    df_plfam_metadata[
        "genome_id"
    ]
    .astype(str)
    .str.strip()
)


df_plfam_metadata = (
    df_plfam_metadata
    .drop_duplicates(
        subset=[
            "genome_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# 6. Merge with exact study cohort
# =========================================================

df_plfam_coverage = (
    df_pangenome_input[
        [
            "Genome ID"
        ]
    ]
    .copy()
)


df_plfam_coverage[
    "Genome ID"
] = (
    df_plfam_coverage[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_plfam_coverage = (
    df_plfam_coverage
    .merge(
        df_plfam_metadata,
        left_on="Genome ID",
        right_on="genome_id",
        how="left"
    )
    .drop(
        columns=[
            "genome_id"
        ]
    )
)


assert len(
    df_plfam_coverage
) == 4233


# =========================================================
# 7. Convert numeric fields
# =========================================================

for col in [
    "patric_cds",
    "plfam_cds",
    "plfam_cds_ratio"
]:

    df_plfam_coverage[col] = pd.to_numeric(
        df_plfam_coverage[col],
        errors="coerce"
    )


# =========================================================
# 8. Independent calculated coverage ratio
# =========================================================

df_plfam_coverage[
    "Calculated PLFam Ratio"
] = np.where(
    df_plfam_coverage[
        "patric_cds"
    ] > 0,

    df_plfam_coverage[
        "plfam_cds"
    ]
    /
    df_plfam_coverage[
        "patric_cds"
    ],

    np.nan
)


# =========================================================
# 9. Coverage thresholds
# =========================================================

valid_ratio = (
    df_plfam_coverage[
        "Calculated PLFam Ratio"
    ]
    .dropna()
)


coverage_90 = int(
    (
        valid_ratio
        >= 0.90
    )
    .sum()
)


coverage_95 = int(
    (
        valid_ratio
        >= 0.95
    )
    .sum()
)


coverage_98 = int(
    (
        valid_ratio
        >= 0.98
    )
    .sum()
)


missing_family_metadata = int(
    df_plfam_coverage[
        "Calculated PLFam Ratio"
    ]
    .isna()
    .sum()
)


# =========================================================
# 10. Save persistent audit
# =========================================================

PLFAM_COVERAGE_PATH = os.path.join(
    PANGENOME_DIR,
    "bvbrc_plfam_coverage_audit.csv"
)

PLFAM_BATCH_AUDIT_PATH = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "bvbrc_plfam_metadata_batch_audit.csv"
)


df_plfam_coverage.to_csv(
    PLFAM_COVERAGE_PATH,
    index=False
)


df_plfam_batch_audit.to_csv(
    PLFAM_BATCH_AUDIT_PATH,
    index=False
)


# =========================================================
# 11. Final summary
# =========================================================

print()
print("=" * 70)
print("BV-BRC PLFAM COVERAGE AUDIT")
print("=" * 70)

print(
    f"Study genomes                  : "
    f"{len(df_plfam_coverage):,}"
)

print(
    f"Genome metadata records        : "
    f"{df_plfam_metadata['genome_id'].nunique():,}"
)

print(
    f"Missing PLFam coverage metadata: "
    f"{missing_family_metadata:,}"
)

print()

print(
    "PLFam assignment ratio:"
)

display(
    valid_ratio
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .to_frame(
        name="PLFam Ratio"
    )
)

print()

print(
    f"Genomes with >=90% gene-family coverage: "
    f"{coverage_90:,}/4,233 "
    f"({coverage_90 / 4233:.2%})"
)

print(
    f"Genomes with >=95% gene-family coverage: "
    f"{coverage_95:,}/4,233 "
    f"({coverage_95 / 4233:.2%})"
)

print(
    f"Genomes with >=98% gene-family coverage: "
    f"{coverage_98:,}/4,233 "
    f"({coverage_98 / 4233:.2%})"
)

print()

print(
    "Lowest-coverage genomes:"
)

display(
    df_plfam_coverage[
        [
            "Genome ID",
            "patric_cds",
            "plfam_cds",
            "Calculated PLFam Ratio"
        ]
    ]
    .sort_values(
        "Calculated PLFam Ratio"
    )
    .head(20)
)

print()

print(
    "Coverage audit saved to:"
)

print(
    PLFAM_COVERAGE_PATH
)

print()

print(
    "PASS: BV-BRC precomputed PLFam coverage "
    "was audited for the full study cohort."
)

BV-BRC PLFAM COVERAGE RETRIEVAL
Study genomes : 4,233
Total batches : 22

[01/22] requested=200 | returned=200 | PASS
[02/22] requested=200 | returned=200 | PASS
[03/22] requested=200 | returned=200 | PASS
[04/22] requested=200 | returned=200 | PASS
[05/22] requested=200 | returned=200 | PASS
[06/22] requested=200 | returned=200 | PASS
[07/22] requested=200 | returned=200 | PASS
[08/22] requested=200 | returned=200 | PASS
[09/22] requested=200 | returned=200 | PASS
[10/22] requested=200 | returned=200 | PASS
[11/22] requested=200 | returned=200 | PASS
[12/22] requested=200 | returned=200 | PASS
[13/22] requested=200 | returned=200 | PASS
[14/22] requested=200 | returned=200 | PASS
[15/22] requested=200 | returned=200 | PASS
[16/22] requested=200 | returned=200 | PASS
[17/22] requested=200 | returned=200 | PASS
[18/22] requested=200 | returned=200 | PASS
[19/22] requested=200 | returned=200 | PASS
[20/22] requested=200 | returned=200 | PASS
[21/22] requested=200 | returned=200 | PASS
[2

,PLFam Ratio
count,4230.000000
mean,0.984454
std,0.015535
min,0.841917
1%,0.956632
5%,0.962140
25%,0.968450
50%,0.992072
75%,0.999439
95%,0.999831



Genomes with >=90% gene-family coverage: 4,229/4,233 (99.91%)
Genomes with >=95% gene-family coverage: 4,227/4,233 (99.86%)
Genomes with >=98% gene-family coverage: 2,345/4,233 (55.40%)

Lowest-coverage genomes:


,Genome ID,patric_cds,plfam_cds,Calculated PLFam Ratio
2527,573.29887,8932,7520.0,0.841917
4184,72407.549,6611,6247.0,0.944940
2756,573.46106,10954,10374.0,0.947051
3257,573.66473,5578,5300.0,0.950161
2576,573.29938,5816,5541.0,0.952717
2947,573.66150,6064,5779.0,0.953001
2761,573.46111,4587,4375.0,0.953782
1983,573.20059,5766,5502.0,0.954214
2791,573.46850,5855,5588.0,0.954398
2745,573.46095,6614,6313.0,0.954490



Coverage audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/bvbrc_plfam_coverage_audit.csv

PASS: BV-BRC precomputed PLFam coverage was audited for the full study cohort.


# Cell 3 — Pilot Retrieval of BV-BRC PLFam Gene-Family Annotations

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd


# =========================================================
# 1. BV-BRC genome-feature API
# =========================================================

BV_BRC_FEATURE_API = (
    "https://www.bv-brc.org/api/genome_feature/"
)

feature_session = requests.Session()

feature_session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": (
            "AMR-Genome-ML/1.0 "
            "(academic research)"
        )
    }
)


# =========================================================
# 2. Build phenotype-independent pilot cohort
# =========================================================

df_cov = (
    df_plfam_coverage
    .copy()
)


valid_cov = (
    df_cov[
        df_cov[
            "Calculated PLFam Ratio"
        ]
        .notna()
    ]
    .sort_values(
        "Calculated PLFam Ratio"
    )
)


pilot_ids = []


# ---------------------------------------------------------
# A. Three genomes with missing coverage metadata
# ---------------------------------------------------------

missing_metadata_ids = (
    df_cov[
        df_cov[
            "Calculated PLFam Ratio"
        ]
        .isna()
    ][
        "Genome ID"
    ]
    .astype(str)
    .tolist()
)


pilot_ids.extend(
    missing_metadata_ids
)


# ---------------------------------------------------------
# B. Three lowest-coverage genomes
# ---------------------------------------------------------

lowest_ids = (
    valid_cov
    .head(3)[
        "Genome ID"
    ]
    .astype(str)
    .tolist()
)


pilot_ids.extend(
    lowest_ids
)


# ---------------------------------------------------------
# C. Two genomes near the median coverage
# ---------------------------------------------------------

median_ratio = (
    valid_cov[
        "Calculated PLFam Ratio"
    ]
    .median()
)


median_ids = (
    valid_cov
    .assign(
        DistanceFromMedian=lambda d:
        (
            d[
                "Calculated PLFam Ratio"
            ]
            - median_ratio
        ).abs()
    )
    .sort_values(
        "DistanceFromMedian"
    )
    .head(2)[
        "Genome ID"
    ]
    .astype(str)
    .tolist()
)


pilot_ids.extend(
    median_ids
)


# ---------------------------------------------------------
# D. Two highest-coverage genomes
# ---------------------------------------------------------

highest_ids = (
    valid_cov
    .tail(2)[
        "Genome ID"
    ]
    .astype(str)
    .tolist()
)


pilot_ids.extend(
    highest_ids
)


# Remove accidental duplicates while preserving order
pilot_ids = list(
    dict.fromkeys(
        pilot_ids
    )
)


print("=" * 70)
print("PLFAM PILOT COHORT")
print("=" * 70)

print(
    f"Pilot genomes: "
    f"{len(pilot_ids)}"
)

print()

display(
    df_cov[
        df_cov[
            "Genome ID"
        ]
        .astype(str)
        .isin(
            pilot_ids
        )
    ][
        [
            "Genome ID",
            "patric_cds",
            "plfam_cds",
            "Calculated PLFam Ratio"
        ]
    ]
    .sort_values(
        "Calculated PLFam Ratio",
        na_position="first"
    )
)


# =========================================================
# 3. Retrieve CDS-level feature annotations
# =========================================================

pilot_feature_records = []

retrieval_audit = []


for i, genome_id in enumerate(
    pilot_ids,
    start=1
):

    query_url = (
        f"{BV_BRC_FEATURE_API}"
        f"?and("
        f"eq(genome_id,{genome_id}),"
        f"eq(feature_type,CDS)"
        f")"
        f"&select("
        f"genome_id,"
        f"feature_id,"
        f"annotation,"
        f"feature_type,"
        f"gene,"
        f"product,"
        f"plfam_id,"
        f"pgfam_id"
        f")"
        f"&limit(20000)"
    )


    start_time = time.time()


    try:

        response = feature_session.get(
            query_url,
            timeout=120
        )

        response.raise_for_status()

        records = response.json()


        if not isinstance(
            records,
            list
        ):

            raise RuntimeError(
                "Unexpected BV-BRC API response format."
            )


        pilot_feature_records.extend(
            records
        )


        elapsed = (
            time.time()
            - start_time
        )


        retrieval_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "PASS",

                "CDS Records":
                    len(records),

                "Runtime Seconds":
                    round(
                        elapsed,
                        2
                    ),

                "Error":
                    None
            }
        )


        print(
            f"[{i:02d}/{len(pilot_ids):02d}] "
            f"{genome_id} | "
            f"{len(records):,} CDS | "
            f"{elapsed:.2f} s"
        )


    except Exception as exc:

        retrieval_audit.append(
            {
                "Genome ID":
                    genome_id,

                "Status":
                    "FAIL",

                "CDS Records":
                    0,

                "Runtime Seconds":
                    None,

                "Error":
                    str(exc)
            }
        )


        print(
            f"[{i:02d}/{len(pilot_ids):02d}] "
            f"{genome_id} | FAIL"
        )


# =========================================================
# 4. Construct pilot tables
# =========================================================

df_plfam_pilot_raw = pd.DataFrame(
    pilot_feature_records
)

df_plfam_pilot_audit = pd.DataFrame(
    retrieval_audit
)


failed = (
    df_plfam_pilot_audit[
        df_plfam_pilot_audit[
            "Status"
        ]
        .ne("PASS")
    ]
)


if not failed.empty:

    display(
        failed
    )

    raise RuntimeError(
        "One or more PLFam pilot retrievals failed."
    )


# =========================================================
# 5. Clean PLFam values
# =========================================================

if not df_plfam_pilot_raw.empty:

    df_plfam_pilot_raw[
        "genome_id"
    ] = (
        df_plfam_pilot_raw[
            "genome_id"
        ]
        .astype(str)
        .str.strip()
    )


    df_plfam_pilot_raw[
        "plfam_id"
    ] = (
        df_plfam_pilot_raw[
            "plfam_id"
        ]
        .replace(
            {
                "":
                    pd.NA,

                "NULL":
                    pd.NA,

                "None":
                    pd.NA,

                "nan":
                    pd.NA
            }
        )
    )


# =========================================================
# 6. Annotation-source audit
# =========================================================

print()
print("=" * 70)
print("PLFAM PILOT ANNOTATION AUDIT")
print("=" * 70)

print(
    f"Total CDS records        : "
    f"{len(df_plfam_pilot_raw):,}"
)

print(
    f"Pilot genomes represented: "
    f"{df_plfam_pilot_raw['genome_id'].nunique():,}"
)


print()
print(
    "Annotation-source distribution:"
)

print(
    df_plfam_pilot_raw[
        "annotation"
    ]
    .value_counts(
        dropna=False
    )
)


# =========================================================
# 7. Calculate per-genome PLFam coverage from feature rows
# =========================================================

pilot_summary_records = []


for genome_id, group in (
    df_plfam_pilot_raw
    .groupby(
        "genome_id"
    )
):

    total_cds = len(
        group
    )


    plfam_cds = int(
        group[
            "plfam_id"
        ]
        .notna()
        .sum()
    )


    unique_plfams = int(
        group[
            "plfam_id"
        ]
        .dropna()
        .astype(str)
        .nunique()
    )


    annotation_types = int(
        group[
            "annotation"
        ]
        .nunique(
            dropna=True
        )
    )


    pilot_summary_records.append(
        {
            "Genome ID":
                genome_id,

            "Retrieved CDS":
                total_cds,

            "CDS with PLFam":
                plfam_cds,

            "Calculated Feature-Level Ratio":
                (
                    plfam_cds
                    / total_cds
                    if total_cds > 0
                    else np.nan
                ),

            "Unique PLFams":
                unique_plfams,

            "Annotation Types":
                annotation_types
        }
    )


df_plfam_pilot_summary = pd.DataFrame(
    pilot_summary_records
)


# Add genome-level metadata from Cell 2
df_plfam_pilot_summary = (
    df_plfam_pilot_summary
    .merge(
        df_cov[
            [
                "Genome ID",
                "patric_cds",
                "plfam_cds",
                "Calculated PLFam Ratio"
            ]
        ],
        on="Genome ID",
        how="left"
    )
)


# =========================================================
# 8. Inspect duplicate family occurrences within genomes
# =========================================================

family_copy_counts = (
    df_plfam_pilot_raw[
        df_plfam_pilot_raw[
            "plfam_id"
        ]
        .notna()
    ]
    .groupby(
        [
            "genome_id",
            "plfam_id"
        ]
    )
    .size()
)


multi_copy_occurrences = int(
    (
        family_copy_counts
        > 1
    )
    .sum()
)


# =========================================================
# 9. Save pilot outputs
# =========================================================

PLFAM_PILOT_RAW_PATH = os.path.join(
    PANGENOME_DIR,
    "plfam_pilot_genome_features.csv"
)

PLFAM_PILOT_SUMMARY_PATH = os.path.join(
    PANGENOME_DIR,
    "plfam_pilot_summary.csv"
)

PLFAM_PILOT_AUDIT_PATH = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "plfam_pilot_retrieval_audit.csv"
)


df_plfam_pilot_raw.to_csv(
    PLFAM_PILOT_RAW_PATH,
    index=False
)


df_plfam_pilot_summary.to_csv(
    PLFAM_PILOT_SUMMARY_PATH,
    index=False
)


df_plfam_pilot_audit.to_csv(
    PLFAM_PILOT_AUDIT_PATH,
    index=False
)


# =========================================================
# 10. Final report
# =========================================================

print()
print(
    "Per-genome pilot summary:"
)

display(
    df_plfam_pilot_summary
    .sort_values(
        "Calculated Feature-Level Ratio",
        na_position="first"
    )
)


print()
print(
    f"Genome-family combinations with "
    f"multiple copies: "
    f"{multi_copy_occurrences:,}"
)


print()
print(
    "Example PLFam-annotated genes:"
)

display(
    df_plfam_pilot_raw[
        df_plfam_pilot_raw[
            "plfam_id"
        ]
        .notna()
    ][
        [
            "genome_id",
            "gene",
            "product",
            "plfam_id",
            "annotation"
        ]
    ]
    .head(20)
)


print()
print(
    "PASS: Gene-level PLFam annotations were "
    "successfully inspected on the pilot cohort."
)

PLFAM PILOT COHORT
Pilot genomes: 10



,Genome ID,patric_cds,plfam_cds,Calculated PLFam Ratio
2702,573.34250,5540,NaN,NaN
2704,573.34254,5538,NaN,NaN
2705,573.34255,5852,NaN,NaN
2527,573.29887,8932,7520.0,0.841917
4184,72407.549,6611,6247.0,0.944940
2756,573.46106,10954,10374.0,0.947051
4142,72407.507,5515,5471.0,0.992022
230,573.12945,5586,5542.0,0.992123
948,573.13713,5408,5408.0,1.000000
1963,573.18087,5834,5834.0,1.000000


[01/10] 573.34250 | 5,540 CDS | 2.34 s
[02/10] 573.34254 | 5,538 CDS | 1.26 s
[03/10] 573.34255 | 5,852 CDS | 1.18 s
[04/10] 573.29887 | 8,932 CDS | 1.60 s
[05/10] 72407.549 | 6,611 CDS | 1.75 s
[06/10] 573.46106 | 10,954 CDS | 1.30 s
[07/10] 573.12945 | 5,586 CDS | 1.47 s
[08/10] 72407.507 | 5,515 CDS | 1.56 s
[09/10] 573.13713 | 5,408 CDS | 1.24 s
[10/10] 573.18087 | 11,666 CDS | 1.67 s

PLFAM PILOT ANNOTATION AUDIT
Total CDS records        : 71,602
Pilot genomes represented: 10

Annotation-source distribution:
annotation
PATRIC    65770
RefSeq     5832
Name: count, dtype: int64

Per-genome pilot summary:


,Genome ID,Retrieved CDS,CDS with PLFam,Calculated Feature-Level Ratio,Unique PLFams,Annotation Types,patric_cds,plfam_cds,Calculated PLFam Ratio
5,573.34254,5538,0,0.000000,0,1,5538,NaN,NaN
6,573.34255,5852,0,0.000000,0,1,5852,NaN,NaN
4,573.34250,5540,0,0.000000,0,1,5540,NaN,NaN
2,573.18087,11666,5834,0.500086,5610,2,5834,5834.0,1.000000
3,573.29887,8932,7520,0.841917,6955,1,8932,7520.0,0.841917
9,72407.549,6611,6247,0.944940,5988,1,6611,6247.0,0.944940
7,573.46106,10954,10374,0.947051,7200,1,10954,10374.0,0.947051
8,72407.507,5515,5471,0.992022,5391,1,5515,5471.0,0.992022
0,573.12945,5586,5542,0.992123,5377,1,5586,5542.0,0.992123
1,573.13713,5408,5408,1.000000,5314,1,5408,5408.0,1.000000



Genome-family combinations with multiple copies: 3,563

Example PLFam-annotated genes:


,genome_id,gene,product,plfam_id,annotation
16931,573.29887,NaN,hypothetical protein,PLF_570_00008367,PATRIC
16933,573.29887,NaN,Dihydroorotase (EC 3.5.2.3),PLF_570_00089222,PATRIC
16934,573.29887,NaN,Lipid carrier : UDP-N-acetylgalactosaminyltran...,PLF_570_00046435,PATRIC
16935,573.29887,NaN,Periplasmic thiol:disulfide interchange protei...,PLF_570_00002407,PATRIC
16937,573.29887,NaN,Anaerobic nitric oxide reductase transcription...,PLF_570_00002995,PATRIC
16939,573.29887,NaN,Phage TraR/YbiI family protein,PLF_570_00036146,PATRIC
16940,573.29887,NaN,"Transcriptional regulator, AraC family",PLF_570_00002765,PATRIC
16941,573.29887,NaN,UPF0758 protein YicR,PLF_570_00003437,PATRIC
16942,573.29887,NaN,Acyl carrier protein phosphodiesterase (EC 3.1...,PLF_570_00002003,PATRIC
16943,573.29887,NaN,Periplasmic protein CpxP,PLF_570_00004408,PATRIC



PASS: Gene-level PLFam annotations were successfully inspected on the pilot cohort.


# Cell 4 — Resolve Annotation Source and Protein-Family Coverage

In [ ]:
import os
import numpy as np
import pandas as pd


# =========================================================
# 1. Keep PATRIC annotation only
# =========================================================

df_plfam_pilot_patric = (
    df_plfam_pilot_raw[
        df_plfam_pilot_raw[
            "annotation"
        ]
        .astype(str)
        .str.upper()
        .eq("PATRIC")
    ]
    .copy()
    .reset_index(drop=True)
)


print("=" * 70)
print("PATRIC-ONLY PROTEIN-FAMILY AUDIT")
print("=" * 70)

print(
    f"All pilot CDS records : "
    f"{len(df_plfam_pilot_raw):,}"
)

print(
    f"PATRIC CDS records    : "
    f"{len(df_plfam_pilot_patric):,}"
)

print(
    f"RefSeq records removed: "
    f"{len(df_plfam_pilot_raw) - len(df_plfam_pilot_patric):,}"
)


# =========================================================
# 2. Clean family identifiers
# =========================================================

for col in [
    "plfam_id",
    "pgfam_id"
]:

    df_plfam_pilot_patric[col] = (
        df_plfam_pilot_patric[col]
        .replace(
            {
                "":
                    pd.NA,

                "NULL":
                    pd.NA,

                "None":
                    pd.NA,

                "nan":
                    pd.NA
            }
        )
    )


# =========================================================
# 3. Recalculate PLFam and PGFam coverage per genome
# =========================================================

family_summary_records = []


for genome_id, group in (
    df_plfam_pilot_patric
    .groupby(
        "genome_id"
    )
):

    total_cds = len(
        group
    )


    plfam_assigned = int(
        group[
            "plfam_id"
        ]
        .notna()
        .sum()
    )


    pgfam_assigned = int(
        group[
            "pgfam_id"
        ]
        .notna()
        .sum()
    )


    unique_plfams = int(
        group[
            "plfam_id"
        ]
        .dropna()
        .astype(str)
        .nunique()
    )


    unique_pgfams = int(
        group[
            "pgfam_id"
        ]
        .dropna()
        .astype(str)
        .nunique()
    )


    family_summary_records.append(
        {
            "Genome ID":
                str(genome_id),

            "PATRIC CDS":
                total_cds,

            "PLFam-assigned CDS":
                plfam_assigned,

            "PLFam Ratio":
                (
                    plfam_assigned
                    / total_cds
                    if total_cds > 0
                    else np.nan
                ),

            "Unique PLFams":
                unique_plfams,

            "PGFam-assigned CDS":
                pgfam_assigned,

            "PGFam Ratio":
                (
                    pgfam_assigned
                    / total_cds
                    if total_cds > 0
                    else np.nan
                ),

            "Unique PGFams":
                unique_pgfams
        }
    )


df_family_strategy_pilot = pd.DataFrame(
    family_summary_records
)


# =========================================================
# 4. Add Cell 2 metadata reference
# =========================================================

df_family_strategy_pilot = (
    df_family_strategy_pilot
    .merge(
        df_plfam_coverage[
            [
                "Genome ID",
                "patric_cds",
                "plfam_cds",
                "Calculated PLFam Ratio"
            ]
        ],
        on="Genome ID",
        how="left"
    )
)


# =========================================================
# 5. Difference between API-derived and metadata PLFam ratio
# =========================================================

df_family_strategy_pilot[
    "PLFam Ratio Difference"
] = (
    df_family_strategy_pilot[
        "PLFam Ratio"
    ]
    -
    df_family_strategy_pilot[
        "Calculated PLFam Ratio"
    ]
).abs()


# =========================================================
# 6. Identify genomes with no PLFam
# =========================================================

df_no_plfam = (
    df_family_strategy_pilot[
        df_family_strategy_pilot[
            "PLFam-assigned CDS"
        ]
        .eq(0)
    ]
    .copy()
)


# =========================================================
# 7. Summary
# =========================================================

print()
print(
    "PATRIC-only family coverage:"
)

display(
    df_family_strategy_pilot[
        [
            "Genome ID",
            "PATRIC CDS",
            "PLFam-assigned CDS",
            "PLFam Ratio",
            "Unique PLFams",
            "PGFam-assigned CDS",
            "PGFam Ratio",
            "Unique PGFams",
            "patric_cds",
            "Calculated PLFam Ratio",
            "PLFam Ratio Difference"
        ]
    ]
    .sort_values(
        "PLFam Ratio"
    )
)


print()
print(
    f"Pilot genomes with no PLFam assignments: "
    f"{len(df_no_plfam)}"
)


if not df_no_plfam.empty:

    display(
        df_no_plfam[
            [
                "Genome ID",
                "PATRIC CDS",
                "PLFam-assigned CDS",
                "PLFam Ratio",
                "PGFam-assigned CDS",
                "PGFam Ratio",
                "Unique PGFams"
            ]
        ]
    )


# =========================================================
# 8. Overall pilot coverage comparison
# =========================================================

print()
print("Pilot coverage comparison:")

print(
    f"Mean PLFam ratio: "
    f"{df_family_strategy_pilot['PLFam Ratio'].mean():.4%}"
)

print(
    f"Mean PGFam ratio: "
    f"{df_family_strategy_pilot['PGFam Ratio'].mean():.4%}"
)

print()

print(
    f"Genomes with PLFam coverage >=95%: "
    f"{(df_family_strategy_pilot['PLFam Ratio'] >= 0.95).sum()}"
    f"/{len(df_family_strategy_pilot)}"
)

print(
    f"Genomes with PGFam coverage >=95%: "
    f"{(df_family_strategy_pilot['PGFam Ratio'] >= 0.95).sum()}"
    f"/{len(df_family_strategy_pilot)}"
)


# =========================================================
# 9. Save strategy audit
# =========================================================

FAMILY_STRATEGY_AUDIT_PATH = os.path.join(
    PANGENOME_DIR,
    "protein_family_strategy_pilot_audit.csv"
)


df_family_strategy_pilot.to_csv(
    FAMILY_STRATEGY_AUDIT_PATH,
    index=False
)


print()
print(
    "Strategy audit saved to:"
)

print(
    FAMILY_STRATEGY_AUDIT_PATH
)

print()

print(
    "PASS: PATRIC-only PLFam and PGFam coverage "
    "were compared on the pilot cohort."
)

PATRIC-ONLY PROTEIN-FAMILY AUDIT
All pilot CDS records : 71,602
PATRIC CDS records    : 65,770
RefSeq records removed: 5,832

PATRIC-only family coverage:


,Genome ID,PATRIC CDS,PLFam-assigned CDS,PLFam Ratio,Unique PLFams,PGFam-assigned CDS,PGFam Ratio,Unique PGFams,patric_cds,Calculated PLFam Ratio,PLFam Ratio Difference
5,573.34254,5538,0,0.000000,0,0,0.000000,0,5538,NaN,NaN
6,573.34255,5852,0,0.000000,0,0,0.000000,0,5852,NaN,NaN
4,573.34250,5540,0,0.000000,0,0,0.000000,0,5540,NaN,NaN
3,573.29887,8932,7520,0.841917,6955,8543,0.956449,5994,8932,0.841917,0.0
9,72407.549,6611,6247,0.944940,5988,6247,0.944940,5105,6611,0.944940,0.0
7,573.46106,10954,10374,0.947051,7200,10560,0.964031,5906,10954,0.947051,0.0
8,72407.507,5515,5471,0.992022,5391,5471,0.992022,4645,5515,0.992022,0.0
0,573.12945,5586,5542,0.992123,5377,5542,0.992123,4600,5586,0.992123,0.0
1,573.13713,5408,5408,1.000000,5314,5408,1.000000,4575,5408,1.000000,0.0
2,573.18087,5834,5834,1.000000,5610,5834,1.000000,4802,5834,1.000000,0.0



Pilot genomes with no PLFam assignments: 3


,Genome ID,PATRIC CDS,PLFam-assigned CDS,PLFam Ratio,PGFam-assigned CDS,PGFam Ratio,Unique PGFams
4,573.34250,5540,0,0.0,0,0.0,0
5,573.34254,5538,0,0.0,0,0.0,0
6,573.34255,5852,0,0.0,0,0.0,0



Pilot coverage comparison:
Mean PLFam ratio: 67.1805%
Mean PGFam ratio: 68.4957%

Genomes with PLFam coverage >=95%: 4/10
Genomes with PGFam coverage >=95%: 6/10

Strategy audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/protein_family_strategy_pilot_audit.csv

PASS: PATRIC-only PLFam and PGFam coverage were compared on the pilot cohort.


# Cell 5 — Resolve Genomes Missing BV-BRC Protein-Family Assignments

In [ ]:
import os
import time
import requests
import pandas as pd
import numpy as np


# =========================================================
# 1. Missing-family genomes identified in Cell 4
# =========================================================

missing_family_ids = (
    df_family_strategy_pilot[
        (
            df_family_strategy_pilot[
                "PLFam-assigned CDS"
            ].eq(0)
        )
        &
        (
            df_family_strategy_pilot[
                "PGFam-assigned CDS"
            ].eq(0)
        )
    ]["Genome ID"]
    .astype(str)
    .tolist()
)


assert set(missing_family_ids) == {
    "573.34250",
    "573.34254",
    "573.34255"
}


print("=" * 70)
print("MISSING PROTEIN-FAMILY RESOLUTION")
print("=" * 70)

print(
    "Target genomes:",
    missing_family_ids
)


# =========================================================
# 2. Load File 04 NCBI accession mapping
# =========================================================

NCBI_MAPPING_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "full_cohort_ncbi_accession_mapping.csv"
)


assert os.path.exists(
    NCBI_MAPPING_PATH
)


df_ncbi_mapping_05 = pd.read_csv(
    NCBI_MAPPING_PATH,
    dtype=str
)


df_ncbi_mapping_05["Genome ID"] = (
    df_ncbi_mapping_05["Genome ID"]
    .astype(str)
    .str.strip()
)


df_missing_mapping = (
    df_ncbi_mapping_05[
        df_ncbi_mapping_05[
            "Genome ID"
        ].isin(
            missing_family_ids
        )
    ]
    .copy()
)


assert len(df_missing_mapping) == 3


print()
print("Existing accession mapping:")

display(
    df_missing_mapping
)


# =========================================================
# 3. Detect accession columns
# =========================================================

def find_column(df, candidates):

    normalized = {
        str(col).strip().lower():
            col
        for col in df.columns
    }

    for candidate in candidates:

        key = candidate.lower()

        if key in normalized:
            return normalized[key]

    return None


assembly_col = find_column(
    df_missing_mapping,
    [
        "Assembly Accession",
        "Study Assembly Accession",
        "assembly_accession"
    ]
)


biosample_col = find_column(
    df_missing_mapping,
    [
        "BioSample Accession",
        "Study BioSample Accession",
        "biosample_accession"
    ]
)


print()
print(
    "Detected assembly column:",
    assembly_col
)

print(
    "Detected BioSample column:",
    biosample_col
)


assert assembly_col is not None
assert biosample_col is not None


# =========================================================
# 4. BV-BRC API
# =========================================================

BV_BRC_GENOME_API = (
    "https://www.bv-brc.org/api/genome/"
)


resolve_session = requests.Session()

resolve_session.headers.update(
    {
        "Accept":
            "application/json",

        "User-Agent":
            "AMR-Genome-ML/1.0 (academic research)"
    }
)


# =========================================================
# 5. Query helper
# =========================================================

def bvbrc_genome_query(filter_expression):

    query_url = (
        f"{BV_BRC_GENOME_API}"
        f"?{filter_expression}"
        f"&select("
        f"genome_id,"
        f"genome_name,"
        f"assembly_accession,"
        f"biosample_accession,"
        f"patric_cds,"
        f"plfam_cds,"
        f"plfam_cds_ratio"
        f")"
        f"&limit(1000)"
    )


    last_error = None


    for attempt in range(1, 4):

        try:

            response = resolve_session.get(
                query_url,
                timeout=120
            )

            response.raise_for_status()

            records = response.json()


            if not isinstance(
                records,
                list
            ):
                raise RuntimeError(
                    "Unexpected BV-BRC response format."
                )


            return records


        except Exception as exc:

            last_error = str(exc)

            if attempt < 3:
                time.sleep(
                    attempt * 2
                )


    raise RuntimeError(
        last_error
    )


# =========================================================
# 6. Search assembly / BioSample candidates
# =========================================================

candidate_records = []


for _, row in df_missing_mapping.iterrows():

    study_genome_id = str(
        row["Genome ID"]
    ).strip()


    assembly = (
        str(
            row[assembly_col]
        ).strip()
        if pd.notna(
            row[assembly_col]
        )
        else None
    )


    biosample = (
        str(
            row[biosample_col]
        ).strip()
        if pd.notna(
            row[biosample_col]
        )
        else None
    )


    if assembly in {
        "",
        "nan",
        "None"
    }:
        assembly = None


    if biosample in {
        "",
        "nan",
        "None"
    }:
        biosample = None


    print()
    print(
        f"Resolving {study_genome_id}"
    )

    print(
        f"  Assembly : {assembly}"
    )

    print(
        f"  BioSample: {biosample}"
    )


    # -----------------------------------------------------
    # Exact assembly
    # -----------------------------------------------------

    if assembly:

        try:

            hits = bvbrc_genome_query(
                f"eq(assembly_accession,{assembly})"
            )

        except Exception as exc:

            print(
                "  Assembly query failed:",
                exc
            )

            hits = []


        print(
            f"  Assembly candidates: "
            f"{len(hits)}"
        )


        for hit in hits:

            record = dict(hit)

            record[
                "Study Genome ID"
            ] = study_genome_id

            record[
                "Study Assembly"
            ] = assembly

            record[
                "Study BioSample"
            ] = biosample

            record[
                "Found By"
            ] = "ASSEMBLY"

            candidate_records.append(
                record
            )


    # -----------------------------------------------------
    # BioSample
    # -----------------------------------------------------

    if biosample:

        try:

            hits = bvbrc_genome_query(
                f"eq(biosample_accession,{biosample})"
            )

        except Exception as exc:

            print(
                "  BioSample query failed:",
                exc
            )

            hits = []


        print(
            f"  BioSample candidates: "
            f"{len(hits)}"
        )


        for hit in hits:

            record = dict(hit)

            record[
                "Study Genome ID"
            ] = study_genome_id

            record[
                "Study Assembly"
            ] = assembly

            record[
                "Study BioSample"
            ] = biosample

            record[
                "Found By"
            ] = "BIOSAMPLE"

            candidate_records.append(
                record
            )


# =========================================================
# 7. Construct candidate DataFrame robustly
# =========================================================

expected_columns = [
    "Study Genome ID",
    "Study Assembly",
    "Study BioSample",
    "Found By",
    "genome_id",
    "genome_name",
    "assembly_accession",
    "biosample_accession",
    "patric_cds",
    "plfam_cds",
    "plfam_cds_ratio"
]


df_family_candidates = pd.DataFrame(
    candidate_records
)


# BV-BRC may omit keys whose values are NULL.
# Force every expected column to exist.
for col in expected_columns:

    if col not in df_family_candidates.columns:

        df_family_candidates[col] = pd.NA


df_family_candidates = (
    df_family_candidates[
        expected_columns
    ]
    .copy()
)


# Remove duplicate candidate records
df_family_candidates = (
    df_family_candidates
    .drop_duplicates(
        subset=[
            "Study Genome ID",
            "genome_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# 8. Numeric conversion
# =========================================================

for col in [
    "patric_cds",
    "plfam_cds",
    "plfam_cds_ratio"
]:

    df_family_candidates[col] = (
        pd.to_numeric(
            df_family_candidates[col],
            errors="coerce"
        )
    )


# =========================================================
# 9. Verify exact assembly
# =========================================================

df_family_candidates[
    "Exact Assembly Verified"
] = (
    df_family_candidates[
        "Study Assembly"
    ]
    .notna()
    &
    df_family_candidates[
        "assembly_accession"
    ]
    .notna()
    &
    (
        df_family_candidates[
            "Study Assembly"
        ]
        .astype(str)
        .str.strip()
        ==
        df_family_candidates[
            "assembly_accession"
        ]
        .astype(str)
        .str.strip()
    )
)


# Missing plfam_cds = no evidence of available PLFam
df_family_candidates[
    "Has PLFam"
] = (
    df_family_candidates[
        "plfam_cds"
    ]
    .fillna(0)
    .gt(0)
)


df_family_candidates[
    "Eligible Exact-Assembly Replacement"
] = (
    df_family_candidates[
        "Exact Assembly Verified"
    ]
    &
    df_family_candidates[
        "Has PLFam"
    ]
)


# =========================================================
# 10. Resolution summary
# =========================================================

resolution_records = []


for genome_id in missing_family_ids:

    candidates = (
        df_family_candidates[
            df_family_candidates[
                "Study Genome ID"
            ]
            .astype(str)
            .eq(
                genome_id
            )
        ]
        .copy()
    )


    eligible = (
        candidates[
            candidates[
                "Eligible Exact-Assembly Replacement"
            ]
        ]
    )


    if len(eligible) == 1:

        status = (
            "EXACT_ASSEMBLY_REPLACEMENT_AVAILABLE"
        )

        replacement_id = (
            eligible.iloc[0][
                "genome_id"
            ]
        )


    elif len(eligible) > 1:

        status = (
            "MULTIPLE_EXACT_ASSEMBLY_REPLACEMENTS"
        )

        replacement_id = pd.NA


    else:

        status = (
            "NO_EXACT_ASSEMBLY_REPLACEMENT"
        )

        replacement_id = pd.NA


    resolution_records.append(
        {
            "Genome ID":
                genome_id,

            "Candidate Records":
                len(candidates),

            "Exact Assembly Candidates":
                int(
                    candidates[
                        "Exact Assembly Verified"
                    ].sum()
                ),

            "Candidates with PLFam":
                int(
                    candidates[
                        "Has PLFam"
                    ].sum()
                ),

            "Eligible Exact Replacements":
                len(eligible),

            "Resolution Status":
                status,

            "Replacement Genome ID":
                replacement_id
        }
    )


df_missing_family_resolution = (
    pd.DataFrame(
        resolution_records
    )
)


# =========================================================
# 11. Save
# =========================================================

FAMILY_CANDIDATE_PATH = os.path.join(
    PANGENOME_DIR,
    "missing_family_bvbrc_candidates.csv"
)

FAMILY_RESOLUTION_PATH = os.path.join(
    PANGENOME_DIR,
    "missing_family_resolution.csv"
)


df_family_candidates.to_csv(
    FAMILY_CANDIDATE_PATH,
    index=False
)


df_missing_family_resolution.to_csv(
    FAMILY_RESOLUTION_PATH,
    index=False
)


# =========================================================
# 12. Report
# =========================================================

print()
print("=" * 70)
print("BV-BRC FAMILY-ANNOTATION RESOLUTION")
print("=" * 70)


print()
print(
    "Candidate BV-BRC records:"
)


if df_family_candidates.empty:

    print(
        "No alternative BV-BRC records found."
    )

else:

    display(
        df_family_candidates[
            [
                "Study Genome ID",
                "genome_id",
                "assembly_accession",
                "biosample_accession",
                "patric_cds",
                "plfam_cds",
                "plfam_cds_ratio",
                "Exact Assembly Verified",
                "Has PLFam",
                "Eligible Exact-Assembly Replacement"
            ]
        ]
        .sort_values(
            [
                "Study Genome ID",
                "Exact Assembly Verified"
            ],
            ascending=[
                True,
                False
            ]
        )
    )


print()
print(
    "Resolution summary:"
)

display(
    df_missing_family_resolution
)


print()
print(
    "PASS: Missing-family genomes were "
    "audited without assuming absent BV-BRC "
    "fields were valid protein-family annotations."
)

MISSING PROTEIN-FAMILY RESOLUTION
Target genomes: ['573.34250', '573.34254', '573.34255']

Existing accession mapping:


,Genome ID,Genome Name,Assembly Accession,BioSample Accession
2702,573.34250,Klebsiella pneumoniae strain INF250-sc-2280154,GCA_904863395.1,SAMEA3357225
2704,573.34254,Klebsiella pneumoniae strain KSB1_1B,GCA_904863435.1,SAMN07738870
2705,573.34255,Klebsiella pneumoniae strain KSB1_6G-sc-2280276,GCA_904864445.1,SAMEA3357345



Detected assembly column: Assembly Accession
Detected BioSample column: BioSample Accession

Resolving 573.34250
  Assembly : GCA_904863395.1
  BioSample: SAMEA3357225
  Assembly candidates: 1
  BioSample candidates: 1

Resolving 573.34254
  Assembly : GCA_904863435.1
  BioSample: SAMN07738870
  Assembly candidates: 1
  BioSample candidates: 1

Resolving 573.34255
  Assembly : GCA_904864445.1
  BioSample: SAMEA3357345
  Assembly candidates: 1
  BioSample candidates: 1

BV-BRC FAMILY-ANNOTATION RESOLUTION

Candidate BV-BRC records:


,Study Genome ID,genome_id,assembly_accession,biosample_accession,patric_cds,plfam_cds,plfam_cds_ratio,Exact Assembly Verified,Has PLFam,Eligible Exact-Assembly Replacement
0,573.34250,573.34250,GCA_904863395.1,SAMEA3357225,5540,NaN,0,True,False,False
1,573.34254,573.34254,GCA_904863435.1,SAMN07738870,5538,NaN,0,True,False,False
2,573.34255,573.34255,GCA_904864445.1,SAMEA3357345,5852,NaN,0,True,False,False



Resolution summary:


,Genome ID,Candidate Records,Exact Assembly Candidates,Candidates with PLFam,Eligible Exact Replacements,Resolution Status,Replacement Genome ID
0,573.34250,1,1,0,0,NO_EXACT_ASSEMBLY_REPLACEMENT,<NA>
1,573.34254,1,1,0,0,NO_EXACT_ASSEMBLY_REPLACEMENT,<NA>
2,573.34255,1,1,0,0,NO_EXACT_ASSEMBLY_REPLACEMENT,<NA>



PASS: Missing-family genomes were audited without assuming absent BV-BRC fields were valid protein-family annotations.


# Cell 6 — Full-Cohort PLFam versus PGFam Coverage Audit

In [ ]:
import os
import time
import requests
import numpy as np
import pandas as pd


# =========================================================
# 1. BV-BRC genome API
# =========================================================

BV_BRC_GENOME_API = (
    "https://www.bv-brc.org/api/genome/"
)


family_session = requests.Session()

family_session.headers.update(
    {
        "Accept":
            "application/json",

        "User-Agent":
            "AMR-Genome-ML/1.0 (academic research)"
    }
)


# =========================================================
# 2. Study Genome IDs
# =========================================================

study_genome_ids = (
    df_pangenome_input[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)


assert len(study_genome_ids) == 4233


# =========================================================
# 3. Batch configuration
# =========================================================

BATCH_SIZE = 200


batches = [
    study_genome_ids[i:i + BATCH_SIZE]
    for i in range(
        0,
        len(study_genome_ids),
        BATCH_SIZE
    )
]


family_records = []
batch_audit = []


print("=" * 70)
print("FULL-COHORT PLFAM / PGFAM COVERAGE RETRIEVAL")
print("=" * 70)

print(
    f"Study genomes : "
    f"{len(study_genome_ids):,}"
)

print(
    f"Total batches : "
    f"{len(batches)}"
)

print()


# =========================================================
# 4. Retrieve family metadata
# =========================================================

for batch_index, batch_ids in enumerate(
    batches,
    start=1
):

    id_string = ",".join(
        batch_ids
    )


    query_url = (
        f"{BV_BRC_GENOME_API}"
        f"?in(genome_id,({id_string}))"
        f"&select("
        f"genome_id,"
        f"patric_cds,"
        f"plfam_cds,"
        f"plfam_cds_ratio,"
        f"pgfam_cds,"
        f"pgfam_cds_ratio"
        f")"
        f"&limit(10000)"
    )


    start_time = time.time()

    success = False
    records = []
    last_error = None


    for attempt in range(
        1,
        4
    ):

        try:

            response = family_session.get(
                query_url,
                timeout=120
            )

            response.raise_for_status()

            records = response.json()


            if not isinstance(
                records,
                list
            ):

                raise RuntimeError(
                    "Unexpected BV-BRC response format."
                )


            family_records.extend(
                records
            )

            success = True

            break


        except Exception as exc:

            last_error = str(
                exc
            )


            if attempt < 3:

                time.sleep(
                    2 * attempt
                )


    elapsed = (
        time.time()
        - start_time
    )


    batch_audit.append(
        {
            "Batch":
                batch_index,

            "Requested":
                len(batch_ids),

            "Returned":
                len(records)
                if success
                else 0,

            "Status":
                "PASS"
                if success
                else "FAIL",

            "Runtime Seconds":
                round(
                    elapsed,
                    2
                ),

            "Error":
                None
                if success
                else last_error
        }
    )


    print(
        f"[{batch_index:02d}/{len(batches):02d}] "
        f"requested={len(batch_ids):3d} | "
        f"returned={len(records) if success else 0:3d} | "
        f"{'PASS' if success else 'FAIL'}"
    )


# =========================================================
# 5. Verify batch retrieval
# =========================================================

df_family_batch_audit = pd.DataFrame(
    batch_audit
)


failed_batches = (
    df_family_batch_audit[
        df_family_batch_audit[
            "Status"
        ]
        .ne("PASS")
    ]
)


if not failed_batches.empty:

    display(
        failed_batches
    )

    raise RuntimeError(
        "One or more BV-BRC batches failed."
    )


# =========================================================
# 6. Construct robust metadata table
# =========================================================

df_family_full = pd.DataFrame(
    family_records
)


expected_columns = [
    "genome_id",
    "patric_cds",
    "plfam_cds",
    "plfam_cds_ratio",
    "pgfam_cds",
    "pgfam_cds_ratio"
]


# BV-BRC may omit fields whose values are NULL.
for col in expected_columns:

    if col not in df_family_full.columns:

        df_family_full[col] = pd.NA


df_family_full = (
    df_family_full[
        expected_columns
    ]
    .copy()
)


df_family_full[
    "genome_id"
] = (
    df_family_full[
        "genome_id"
    ]
    .astype(str)
    .str.strip()
)


df_family_full = (
    df_family_full
    .drop_duplicates(
        subset=[
            "genome_id"
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    df_family_full[
        "genome_id"
    ]
    .nunique()
    == 4233
)


# =========================================================
# 7. Numeric conversion
# =========================================================

for col in [
    "patric_cds",
    "plfam_cds",
    "plfam_cds_ratio",
    "pgfam_cds",
    "pgfam_cds_ratio"
]:

    df_family_full[col] = pd.to_numeric(
        df_family_full[col],
        errors="coerce"
    )


# =========================================================
# 8. Merge to exact study cohort
# =========================================================

df_family_full_coverage = (
    df_pangenome_input[
        [
            "Genome ID"
        ]
    ]
    .copy()
)


df_family_full_coverage[
    "Genome ID"
] = (
    df_family_full_coverage[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_family_full_coverage = (
    df_family_full_coverage
    .merge(
        df_family_full,
        left_on="Genome ID",
        right_on="genome_id",
        how="left"
    )
    .drop(
        columns=[
            "genome_id"
        ]
    )
)


assert len(
    df_family_full_coverage
) == 4233


# =========================================================
# 9. Calculate coverage independently
# =========================================================

df_family_full_coverage[
    "PLFam Ratio"
] = np.where(
    df_family_full_coverage[
        "patric_cds"
    ] > 0,

    df_family_full_coverage[
        "plfam_cds"
    ].fillna(0)
    /
    df_family_full_coverage[
        "patric_cds"
    ],

    np.nan
)


df_family_full_coverage[
    "PGFam Ratio"
] = np.where(
    df_family_full_coverage[
        "patric_cds"
    ] > 0,

    df_family_full_coverage[
        "pgfam_cds"
    ].fillna(0)
    /
    df_family_full_coverage[
        "patric_cds"
    ],

    np.nan
)


# =========================================================
# 10. Identify family-assignment failures
# =========================================================

df_family_full_coverage[
    "No PLFam"
] = (
    df_family_full_coverage[
        "plfam_cds"
    ]
    .fillna(0)
    .eq(0)
)


df_family_full_coverage[
    "No PGFam"
] = (
    df_family_full_coverage[
        "pgfam_cds"
    ]
    .fillna(0)
    .eq(0)
)


df_family_full_coverage[
    "No Protein Families"
] = (
    df_family_full_coverage[
        "No PLFam"
    ]
    &
    df_family_full_coverage[
        "No PGFam"
    ]
)


# =========================================================
# 11. Compare representations genome-by-genome
# =========================================================

df_family_full_coverage[
    "PGFam Minus PLFam"
] = (
    df_family_full_coverage[
        "PGFam Ratio"
    ]
    -
    df_family_full_coverage[
        "PLFam Ratio"
    ]
)


pgfam_better = int(
    (
        df_family_full_coverage[
            "PGFam Minus PLFam"
        ]
        > 1e-12
    )
    .sum()
)


plfam_better = int(
    (
        df_family_full_coverage[
            "PGFam Minus PLFam"
        ]
        < -1e-12
    )
    .sum()
)


equal_coverage = int(
    (
        df_family_full_coverage[
            "PGFam Minus PLFam"
        ]
        .abs()
        <= 1e-12
    )
    .sum()
)


# =========================================================
# 12. Coverage threshold helper
# =========================================================

def coverage_counts(series):

    return {
        ">=90%":
            int(
                (series >= 0.90).sum()
            ),

        ">=95%":
            int(
                (series >= 0.95).sum()
            ),

        ">=98%":
            int(
                (series >= 0.98).sum()
            ),

        ">=99%":
            int(
                (series >= 0.99).sum()
            )
    }


plfam_thresholds = coverage_counts(
    df_family_full_coverage[
        "PLFam Ratio"
    ]
)


pgfam_thresholds = coverage_counts(
    df_family_full_coverage[
        "PGFam Ratio"
    ]
)


# =========================================================
# 13. Save persistent audit
# =========================================================

FULL_FAMILY_COVERAGE_PATH = os.path.join(
    PANGENOME_DIR,
    "full_cohort_plfam_pgfam_coverage_audit.csv"
)


FULL_FAMILY_BATCH_AUDIT_PATH = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "full_cohort_family_metadata_batch_audit.csv"
)


df_family_full_coverage.to_csv(
    FULL_FAMILY_COVERAGE_PATH,
    index=False
)


df_family_batch_audit.to_csv(
    FULL_FAMILY_BATCH_AUDIT_PATH,
    index=False
)


# =========================================================
# 14. Report
# =========================================================

print()
print("=" * 70)
print("FULL-COHORT PROTEIN-FAMILY COVERAGE AUDIT")
print("=" * 70)

print(
    f"Study genomes              : "
    f"{len(df_family_full_coverage):,}"
)

print(
    f"No PLFam assignments       : "
    f"{df_family_full_coverage['No PLFam'].sum():,}"
)

print(
    f"No PGFam assignments       : "
    f"{df_family_full_coverage['No PGFam'].sum():,}"
)

print(
    f"No protein-family assignment: "
    f"{df_family_full_coverage['No Protein Families'].sum():,}"
)


print()
print("PLFam coverage:")

display(
    df_family_full_coverage[
        "PLFam Ratio"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .to_frame(
        name="PLFam Ratio"
    )
)


print()
print("PGFam coverage:")

display(
    df_family_full_coverage[
        "PGFam Ratio"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
    .to_frame(
        name="PGFam Ratio"
    )
)


print()
print("Coverage thresholds:")

df_threshold_report = pd.DataFrame(
    {
        "PLFam":
            plfam_thresholds,

        "PGFam":
            pgfam_thresholds
    }
)

display(
    df_threshold_report
)


print()
print("Genome-level comparison:")

print(
    f"PGFam higher coverage : "
    f"{pgfam_better:,}"
)

print(
    f"PLFam higher coverage : "
    f"{plfam_better:,}"
)

print(
    f"Equal coverage        : "
    f"{equal_coverage:,}"
)


print()
print(
    "Genomes lacking both PLFam and PGFam:"
)

display(
    df_family_full_coverage[
        df_family_full_coverage[
            "No Protein Families"
        ]
    ][
        [
            "Genome ID",
            "patric_cds",
            "plfam_cds",
            "pgfam_cds",
            "PLFam Ratio",
            "PGFam Ratio"
        ]
    ]
)


print()
print(
    "Largest PGFam coverage improvements over PLFam:"
)

display(
    df_family_full_coverage[
        [
            "Genome ID",
            "patric_cds",
            "PLFam Ratio",
            "PGFam Ratio",
            "PGFam Minus PLFam"
        ]
    ]
    .sort_values(
        "PGFam Minus PLFam",
        ascending=False
    )
    .head(20)
)


print()
print(
    "Audit saved to:"
)

print(
    FULL_FAMILY_COVERAGE_PATH
)

print()

print(
    "PASS: PLFam and PGFam coverage were compared "
    "across all 4,233 study genomes."
)

FULL-COHORT PLFAM / PGFAM COVERAGE RETRIEVAL
Study genomes : 4,233
Total batches : 22

[01/22] requested=200 | returned=200 | PASS
[02/22] requested=200 | returned=200 | PASS
[03/22] requested=200 | returned=200 | PASS
[04/22] requested=200 | returned=200 | PASS
[05/22] requested=200 | returned=200 | PASS
[06/22] requested=200 | returned=200 | PASS
[07/22] requested=200 | returned=200 | PASS
[08/22] requested=200 | returned=200 | PASS
[09/22] requested=200 | returned=200 | PASS
[10/22] requested=200 | returned=200 | PASS
[11/22] requested=200 | returned=200 | PASS
[12/22] requested=200 | returned=200 | PASS
[13/22] requested=200 | returned=200 | PASS
[14/22] requested=200 | returned=200 | PASS
[15/22] requested=200 | returned=200 | PASS
[16/22] requested=200 | returned=200 | PASS
[17/22] requested=200 | returned=200 | PASS
[18/22] requested=200 | returned=200 | PASS
[19/22] requested=200 | returned=200 | PASS
[20/22] requested=200 | returned=200 | PASS
[21/22] requested=200 | returned=

,PLFam Ratio
count,4233.000000
mean,0.983757
std,0.030458
min,0.000000
1%,0.956499
5%,0.962049
25%,0.968421
50%,0.991859
75%,0.999439
95%,0.999831



PGFam coverage:


,PGFam Ratio
count,4233.0
mean,0.0
std,0.0
min,0.0
1%,0.0
5%,0.0
25%,0.0
50%,0.0
75%,0.0
95%,0.0



Coverage thresholds:


...


Genome-level comparison:
PGFam higher coverage : 0
PLFam higher coverage : 4,230
Equal coverage        : 3

Genomes lacking both PLFam and PGFam:


,Genome ID,patric_cds,plfam_cds,pgfam_cds,PLFam Ratio,PGFam Ratio
2702,573.34250,5540,NaN,NaN,0.0,0.0
2704,573.34254,5538,NaN,NaN,0.0,0.0
2705,573.34255,5852,NaN,NaN,0.0,0.0



Largest PGFam coverage improvements over PLFam:


,Genome ID,patric_cds,PLFam Ratio,PGFam Ratio,PGFam Minus PLFam
2702,573.34250,5540,0.000000,0.0,0.000000
2705,573.34255,5852,0.000000,0.0,0.000000
2704,573.34254,5538,0.000000,0.0,0.000000
2527,573.29887,8932,0.841917,0.0,-0.841917
4184,72407.549,6611,0.944940,0.0,-0.944940
2756,573.46106,10954,0.947051,0.0,-0.947051
3257,573.66473,5578,0.950161,0.0,-0.950161
2576,573.29938,5816,0.952717,0.0,-0.952717
2947,573.66150,6064,0.953001,0.0,-0.953001
2761,573.46111,4587,0.953782,0.0,-0.953782



Audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/full_cohort_plfam_pgfam_coverage_audit.csv

PASS: PLFam and PGFam coverage were compared across all 4,233 study genomes.


# Cell 7 — Finalize the PLFam Pangenome Analysis Cohort

In [ ]:
import os
import pandas as pd
import numpy as np


# =========================================================
# 1. Technical completeness threshold
# =========================================================

PLFAM_MIN_COVERAGE = 0.95


# =========================================================
# 2. Start from full-cohort PLFam coverage audit
# =========================================================

df_pangenome_cohort = (
    df_family_full_coverage[
        [
            "Genome ID",
            "patric_cds",
            "plfam_cds",
            "PLFam Ratio"
        ]
    ]
    .copy()
)


df_pangenome_cohort[
    "Genome ID"
] = (
    df_pangenome_cohort[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_pangenome_cohort
) == 4233


assert (
    df_pangenome_cohort[
        "Genome ID"
    ]
    .nunique()
    == 4233
)


# =========================================================
# 3. Apply phenotype-independent PLFam completeness rule
# =========================================================

df_pangenome_cohort[
    "Pangenome Eligible"
] = (
    df_pangenome_cohort[
        "PLFam Ratio"
    ]
    .ge(
        PLFAM_MIN_COVERAGE
    )
)


df_pangenome_cohort[
    "Exclusion Reason"
] = np.where(
    df_pangenome_cohort[
        "Pangenome Eligible"
    ],
    pd.NA,
    "PLFam coverage <95%"
)


# =========================================================
# 4. Split retained / excluded cohorts
# =========================================================

df_pangenome_retained = (
    df_pangenome_cohort[
        df_pangenome_cohort[
            "Pangenome Eligible"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


df_pangenome_excluded = (
    df_pangenome_cohort[
        ~df_pangenome_cohort[
            "Pangenome Eligible"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert len(
    df_pangenome_retained
) == 4227, (
    f"Expected 4,227 retained genomes, "
    f"found {len(df_pangenome_retained):,}."
)


assert len(
    df_pangenome_excluded
) == 6, (
    f"Expected 6 excluded genomes, "
    f"found {len(df_pangenome_excluded):,}."
)


# =========================================================
# 5. Add phenotype ONLY AFTER cohort decision
#    for exclusion-bias audit
# =========================================================

phenotype_lookup = (
    df_pangenome_input[
        [
            "Genome ID",
            "Resistant Phenotype"
        ]
    ]
    .copy()
)


phenotype_lookup[
    "Genome ID"
] = (
    phenotype_lookup[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_pangenome_cohort = (
    df_pangenome_cohort
    .merge(
        phenotype_lookup,
        on="Genome ID",
        how="left"
    )
)


df_pangenome_retained = (
    df_pangenome_retained
    .merge(
        phenotype_lookup,
        on="Genome ID",
        how="left"
    )
)


df_pangenome_excluded = (
    df_pangenome_excluded
    .merge(
        phenotype_lookup,
        on="Genome ID",
        how="left"
    )
)


assert (
    df_pangenome_cohort[
        "Resistant Phenotype"
    ]
    .notna()
    .all()
)


# =========================================================
# 6. Phenotype exclusion audit
# =========================================================

full_counts = (
    df_pangenome_cohort[
        "Resistant Phenotype"
    ]
    .value_counts()
)


retained_counts = (
    df_pangenome_retained[
        "Resistant Phenotype"
    ]
    .value_counts()
)


excluded_counts = (
    df_pangenome_excluded[
        "Resistant Phenotype"
    ]
    .value_counts()
)


# =========================================================
# 7. Retention rates by phenotype
# =========================================================

phenotype_audit_records = []


for phenotype in [
    "Resistant",
    "Susceptible"
]:

    total = int(
        full_counts.get(
            phenotype,
            0
        )
    )

    retained = int(
        retained_counts.get(
            phenotype,
            0
        )
    )

    excluded = int(
        excluded_counts.get(
            phenotype,
            0
        )
    )


    phenotype_audit_records.append(
        {
            "Phenotype":
                phenotype,

            "Total":
                total,

            "Retained":
                retained,

            "Excluded":
                excluded,

            "Retention Rate":
                (
                    retained / total
                    if total > 0
                    else np.nan
                ),

            "Exclusion Rate":
                (
                    excluded / total
                    if total > 0
                    else np.nan
                )
        }
    )


df_pangenome_phenotype_audit = pd.DataFrame(
    phenotype_audit_records
)


# =========================================================
# 8. Create final File 05 input manifest
# =========================================================

df_pangenome_manifest = (
    df_pangenome_input
    .copy()
)


df_pangenome_manifest[
    "Genome ID"
] = (
    df_pangenome_manifest[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


df_pangenome_manifest = (
    df_pangenome_manifest
    .merge(
        df_pangenome_retained[
            [
                "Genome ID",
                "PLFam Ratio"
            ]
        ],
        on="Genome ID",
        how="inner"
    )
)


assert len(
    df_pangenome_manifest
) == 4227


assert (
    df_pangenome_manifest[
        "Genome ID"
    ]
    .nunique()
    == 4227
)


assert (
    df_pangenome_manifest[
        "FASTA Exists"
    ]
    .all()
)


# =========================================================
# 9. Persistent outputs
# =========================================================

PANGENOME_MANIFEST_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_analysis_manifest.csv"
)

PANGENOME_EXCLUDED_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_technical_exclusions.csv"
)

PANGENOME_PHENOTYPE_AUDIT_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_exclusion_phenotype_audit.csv"
)

PANGENOME_STRATEGY_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_strategy_decision.csv"
)


df_pangenome_manifest.to_csv(
    PANGENOME_MANIFEST_PATH,
    index=False
)


df_pangenome_excluded.to_csv(
    PANGENOME_EXCLUDED_PATH,
    index=False
)


df_pangenome_phenotype_audit.to_csv(
    PANGENOME_PHENOTYPE_AUDIT_PATH,
    index=False
)


# =========================================================
# 10. Record strategy decision
# =========================================================

df_pangenome_strategy = pd.DataFrame(
    [
        {
            "Representation":
                "BV-BRC PLFam presence/absence",

            "Annotation Source":
                "PATRIC",

            "Minimum PLFam Coverage":
                PLFAM_MIN_COVERAGE,

            "Original Study Genomes":
                4233,

            "Pangenome Genomes":
                4227,

            "Technical Exclusions":
                6,

            "Phenotype Used for Eligibility":
                False,

            "PGFam Metadata Used":
                False,

            "Reason":
                (
                    "PLFam metadata were available and highly "
                    "complete across the cohort; genomes below "
                    "95% family-assignment coverage were excluded "
                    "to reduce false gene-family absences."
                )
        }
    ]
)


df_pangenome_strategy.to_csv(
    PANGENOME_STRATEGY_PATH,
    index=False
)


# =========================================================
# 11. Report
# =========================================================

print("=" * 70)
print("PANGENOME COHORT FINALIZATION")
print("=" * 70)

print(
    f"Original study genomes     : "
    f"{len(df_pangenome_cohort):,}"
)

print(
    f"PLFam coverage threshold   : "
    f"{PLFAM_MIN_COVERAGE:.0%}"
)

print(
    f"Pangenome genomes retained : "
    f"{len(df_pangenome_retained):,}"
)

print(
    f"Technical exclusions       : "
    f"{len(df_pangenome_excluded):,}"
)

print(
    f"Retention                  : "
    f"{len(df_pangenome_retained) / 4233:.2%}"
)

print()


print(
    "Excluded genomes:"
)

display(
    df_pangenome_excluded[
        [
            "Genome ID",
            "patric_cds",
            "plfam_cds",
            "PLFam Ratio",
            "Resistant Phenotype",
            "Exclusion Reason"
        ]
    ]
    .sort_values(
        "PLFam Ratio"
    )
)


print()
print(
    "Phenotype exclusion audit:"
)

display(
    df_pangenome_phenotype_audit
)


print()
print(
    "Retained PLFam coverage:"
)

display(
    df_pangenome_retained[
        "PLFam Ratio"
    ]
    .describe()
    .to_frame(
        name="PLFam Ratio"
    )
)


print()
print(
    "Pangenome manifest saved to:"
)

print(
    PANGENOME_MANIFEST_PATH
)

print()

print(
    "PASS: The primary PLFam pangenome cohort "
    "was finalized with 4,227 genomes using a "
    "phenotype-independent >=95% annotation-coverage rule."
)

PANGENOME COHORT FINALIZATION
Original study genomes     : 4,233
PLFam coverage threshold   : 95%
Pangenome genomes retained : 4,227
Technical exclusions       : 6
Retention                  : 99.86%

Excluded genomes:


,Genome ID,patric_cds,plfam_cds,PLFam Ratio,Resistant Phenotype,Exclusion Reason
1,573.34250,5540,NaN,0.000000,Susceptible,PLFam coverage <95%
2,573.34254,5538,NaN,0.000000,Susceptible,PLFam coverage <95%
3,573.34255,5852,NaN,0.000000,Susceptible,PLFam coverage <95%
0,573.29887,8932,7520.0,0.841917,Resistant,PLFam coverage <95%
5,72407.549,6611,6247.0,0.944940,Resistant,PLFam coverage <95%
4,573.46106,10954,10374.0,0.947051,Susceptible,PLFam coverage <95%



Phenotype exclusion audit:


,Phenotype,Total,Retained,Excluded,Retention Rate,Exclusion Rate
0,Resistant,1603,1601,2,0.998752,0.001248
1,Susceptible,2630,2626,4,0.998479,0.001521



Retained PLFam coverage:


,PLFam Ratio
count,4227.000000
mean,0.984506
std,0.015362
min,0.950161
25%,0.968460
50%,0.992221
75%,0.999439
max,1.000000



Pangenome manifest saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/pangenome_analysis_manifest.csv

PASS: The primary PLFam pangenome cohort was finalized with 4,227 genomes using a phenotype-independent >=95% annotation-coverage rule.


# Cell 8 — Retrieve PLFam Membership for the Pangenome Cohort

In [10]:
import os
import time
import random
import threading
import requests
import pandas as pd
import numpy as np

from concurrent.futures import (
    ThreadPoolExecutor,
    as_completed
)


# =========================================================
# 1. Persistent paths
# =========================================================

PANGENOME_MANIFEST_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_analysis_manifest.csv"
)

FULL_FAMILY_COVERAGE_PATH = os.path.join(
    PANGENOME_DIR,
    "full_cohort_plfam_pgfam_coverage_audit.csv"
)

PLFAM_BATCH_DIR = os.path.join(
    PANGENOME_DIR,
    "plfam_membership_batches"
)

PLFAM_BATCH_AUDIT_DIR = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "plfam_membership_batch_audits"
)

PLFAM_MASTER_AUDIT_PATH = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "plfam_membership_retrieval_audit.csv"
)


for path in [
    PLFAM_BATCH_DIR,
    PLFAM_BATCH_AUDIT_DIR
]:

    os.makedirs(
        path,
        exist_ok=True
    )


# =========================================================
# 2. Reload finalized cohort
# =========================================================

df_pangenome_manifest_08 = pd.read_csv(
    PANGENOME_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_pangenome_manifest_08["Genome ID"] = (
    df_pangenome_manifest_08[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_pangenome_manifest_08
) == 4227


assert (
    df_pangenome_manifest_08[
        "Genome ID"
    ]
    .nunique()
    == 4227
)


# =========================================================
# 3. Reload genome-level BV-BRC reference counts
# =========================================================

df_family_coverage_08 = pd.read_csv(
    FULL_FAMILY_COVERAGE_PATH,
    dtype={
        "Genome ID": str
    }
)


df_family_coverage_08["Genome ID"] = (
    df_family_coverage_08[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


expected_counts = (
    df_family_coverage_08[
        df_family_coverage_08[
            "Genome ID"
        ]
        .isin(
            df_pangenome_manifest_08[
                "Genome ID"
            ]
        )
    ][
        [
            "Genome ID",
            "patric_cds",
            "plfam_cds",
            "PLFam Ratio"
        ]
    ]
    .copy()
)


for col in [
    "patric_cds",
    "plfam_cds",
    "PLFam Ratio"
]:

    expected_counts[col] = pd.to_numeric(
        expected_counts[col],
        errors="coerce"
    )


assert len(
    expected_counts
) == 4227


# =========================================================
# 4. Retrieval configuration
# =========================================================

BV_BRC_FEATURE_API = (
    "https://www.bv-brc.org/api/genome_feature/"
)

API_LIMIT = 20000

BATCH_SIZE = 50

MAX_WORKERS = 4

MAX_RETRIES = 4

MIN_OBSERVED_PLFAM_COVERAGE = 0.95


study_ids = (
    df_pangenome_manifest_08[
        "Genome ID"
    ]
    .tolist()
)


batches = [
    study_ids[
        i:i + BATCH_SIZE
    ]
    for i in range(
        0,
        len(study_ids),
        BATCH_SIZE
    )
]


expected_lookup = (
    expected_counts
    .set_index(
        "Genome ID"
    )
    .to_dict(
        orient="index"
    )
)


print("=" * 70)
print("PLFAM MEMBERSHIP RETRIEVAL — RESUME")
print("=" * 70)

print(
    f"Pangenome genomes       : "
    f"{len(study_ids):,}"
)

print(
    f"Batch size              : "
    f"{BATCH_SIZE}"
)

print(
    f"Total batches           : "
    f"{len(batches)}"
)

print(
    f"Concurrent requests     : "
    f"{MAX_WORKERS}"
)

print(
    f"Minimum observed coverage: "
    f"{MIN_OBSERVED_PLFAM_COVERAGE:.0%}"
)

print()


# =========================================================
# 5. Thread-local HTTP sessions
# =========================================================

thread_local = threading.local()


def get_session():

    if not hasattr(
        thread_local,
        "session"
    ):

        session = requests.Session()

        session.headers.update(
            {
                "Accept":
                    "application/json",

                "User-Agent":
                    (
                        "AMR-Genome-ML/1.0 "
                        "(academic research)"
                    )
            }
        )

        thread_local.session = session


    return thread_local.session


# =========================================================
# 6. Single-genome retrieval
# =========================================================

def retrieve_plfam_for_genome(
    genome_id
):

    reference = expected_lookup[
        genome_id
    ]


    metadata_cds = int(
        reference[
            "patric_cds"
        ]
    )


    metadata_plfam_cds = int(
        reference[
            "plfam_cds"
        ]
    )


    metadata_ratio = float(
        reference[
            "PLFam Ratio"
        ]
    )


    query_url = (
        f"{BV_BRC_FEATURE_API}"
        f"?and("
        f"eq(genome_id,{genome_id}),"
        f"eq(feature_type,CDS),"
        f"eq(annotation,PATRIC)"
        f")"
        f"&select("
        f"genome_id,"
        f"plfam_id"
        f")"
        f"&limit({API_LIMIT})"
    )


    last_error = None


    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        start_time = time.time()


        try:

            session = get_session()


            response = session.get(
                query_url,
                timeout=120
            )

            response.raise_for_status()


            records = response.json()


            if not isinstance(
                records,
                list
            ):

                raise RuntimeError(
                    "Unexpected BV-BRC API response."
                )


            returned_cds = len(
                records
            )


            # ---------------------------------------------
            # Detect possible truncation
            # ---------------------------------------------

            if returned_cds >= API_LIMIT:

                raise RuntimeError(
                    "API response reached the configured limit; "
                    "possible truncation."
                )


            # ---------------------------------------------
            # Extract valid PLFam IDs
            # ---------------------------------------------

            plfam_values = []


            for record in records:

                value = record.get(
                    "plfam_id"
                )


                if value is None:
                    continue


                value = str(
                    value
                ).strip()


                if value in {
                    "",
                    "NULL",
                    "None",
                    "nan"
                }:

                    continue


                plfam_values.append(
                    value
                )


            returned_plfam_cds = len(
                plfam_values
            )


            # ---------------------------------------------
            # Observed feature-level coverage
            # ---------------------------------------------

            observed_ratio = (
                returned_plfam_cds
                / returned_cds
                if returned_cds > 0
                else 0.0
            )


            if observed_ratio < MIN_OBSERVED_PLFAM_COVERAGE:

                raise RuntimeError(
                    f"Observed PLFam coverage below threshold: "
                    f"{observed_ratio:.4f}"
                )


            # ---------------------------------------------
            # Metadata/API differences are audited,
            # not treated as automatic failures
            # ---------------------------------------------

            cds_difference = (
                returned_cds
                - metadata_cds
            )


            plfam_cds_difference = (
                returned_plfam_cds
                - metadata_plfam_cds
            )


            ratio_difference = (
                observed_ratio
                - metadata_ratio
            )


            # ---------------------------------------------
            # Collapse to genome-family copy counts
            # ---------------------------------------------

            family_counts = (
                pd.Series(
                    plfam_values,
                    dtype="string"
                )
                .value_counts()
            )


            family_rows = [
                {
                    "Genome ID":
                        genome_id,

                    "PLFam ID":
                        family_id,

                    "Copy Number":
                        int(copy_number)
                }

                for family_id, copy_number
                in family_counts.items()
            ]


            elapsed = (
                time.time()
                - start_time
            )


            audit = {
                "Genome ID":
                    genome_id,

                "Status":
                    "PASS",

                "Metadata CDS":
                    metadata_cds,

                "Returned CDS":
                    returned_cds,

                "CDS Difference":
                    cds_difference,

                "Metadata PLFam CDS":
                    metadata_plfam_cds,

                "Returned PLFam CDS":
                    returned_plfam_cds,

                "PLFam CDS Difference":
                    plfam_cds_difference,

                "Metadata PLFam Ratio":
                    metadata_ratio,

                "Observed PLFam Ratio":
                    observed_ratio,

                "Ratio Difference":
                    ratio_difference,

                "Unique PLFams":
                    len(
                        family_counts
                    ),

                "Runtime Seconds":
                    round(
                        elapsed,
                        3
                    ),

                "Attempts":
                    attempt,

                "Error":
                    None
            }


            return (
                family_rows,
                audit
            )


        except Exception as exc:

            last_error = str(
                exc
            )


            if attempt < MAX_RETRIES:

                time.sleep(
                    (
                        2 ** (
                            attempt - 1
                        )
                    )
                    +
                    random.uniform(
                        0.1,
                        0.8
                    )
                )


    return (
        [],
        {
            "Genome ID":
                genome_id,

            "Status":
                "FAIL",

            "Metadata CDS":
                metadata_cds,

            "Returned CDS":
                pd.NA,

            "CDS Difference":
                pd.NA,

            "Metadata PLFam CDS":
                metadata_plfam_cds,

            "Returned PLFam CDS":
                pd.NA,

            "PLFam CDS Difference":
                pd.NA,

            "Metadata PLFam Ratio":
                metadata_ratio,

            "Observed PLFam Ratio":
                pd.NA,

            "Ratio Difference":
                pd.NA,

            "Unique PLFams":
                pd.NA,

            "Runtime Seconds":
                pd.NA,

            "Attempts":
                MAX_RETRIES,

            "Error":
                last_error
        }
    )


# =========================================================
# 7. Resume-safe batch processing
# =========================================================

start_all = time.time()

completed_batches = 0

new_batches = 0


for batch_number, batch_ids in enumerate(
    batches,
    start=1
):

    batch_data_path = os.path.join(
        PLFAM_BATCH_DIR,
        f"plfam_membership_batch_{batch_number:03d}.csv.gz"
    )


    batch_audit_path = os.path.join(
        PLFAM_BATCH_AUDIT_DIR,
        f"plfam_membership_audit_{batch_number:03d}.csv"
    )


    # -----------------------------------------------------
    # Skip completed batches
    # -----------------------------------------------------

    if (
        os.path.exists(
            batch_data_path
        )
        and
        os.path.exists(
            batch_audit_path
        )
    ):

        existing_audit = pd.read_csv(
            batch_audit_path,
            dtype={
                "Genome ID": str
            }
        )


        audited_ids = set(
            existing_audit[
                "Genome ID"
            ]
            .astype(str)
            .str.strip()
        )


        if (
            len(existing_audit)
            == len(batch_ids)
            and
            audited_ids
            == set(batch_ids)
            and
            existing_audit[
                "Status"
            ]
            .eq("PASS")
            .all()
        ):

            completed_batches += 1


            print(
                f"[{batch_number:03d}/{len(batches):03d}] "
                f"SKIP — already complete"
            )


            continue


    # -----------------------------------------------------
    # Retrieve current batch
    # -----------------------------------------------------

    batch_start = time.time()

    batch_family_rows = []

    batch_audit_rows = []


    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                retrieve_plfam_for_genome,
                genome_id
            ):
            genome_id

            for genome_id in batch_ids
        }


        for future in as_completed(
            futures
        ):

            family_rows, audit = (
                future.result()
            )


            batch_family_rows.extend(
                family_rows
            )

            batch_audit_rows.append(
                audit
            )


    df_batch_audit = pd.DataFrame(
        batch_audit_rows
    )


    df_batch_audit["Genome ID"] = (
        df_batch_audit[
            "Genome ID"
        ]
        .astype(str)
        .str.strip()
    )


    df_batch_audit = (
        df_batch_audit
        .sort_values(
            "Genome ID"
        )
        .reset_index(
            drop=True
        )
    )


    # Always save audit first
    df_batch_audit.to_csv(
        batch_audit_path,
        index=False
    )


    failed = (
        df_batch_audit[
            df_batch_audit[
                "Status"
            ]
            .ne("PASS")
        ]
    )


    if not failed.empty:

        print()
        print(
            f"Batch {batch_number} contains failures:"
        )

        display(
            failed
        )


        raise RuntimeError(
            f"Retrieval stopped at batch "
            f"{batch_number}. "
            f"Earlier completed batches remain saved."
        )


    # -----------------------------------------------------
    # Save compact membership table
    # -----------------------------------------------------

    df_batch_membership = pd.DataFrame(
        batch_family_rows,
        columns=[
            "Genome ID",
            "PLFam ID",
            "Copy Number"
        ]
    )


    df_batch_membership = (
        df_batch_membership
        .sort_values(
            [
                "Genome ID",
                "PLFam ID"
            ]
        )
        .reset_index(
            drop=True
        )
    )


    df_batch_membership.to_csv(
        batch_data_path,
        index=False,
        compression="gzip"
    )


    completed_batches += 1

    new_batches += 1


    elapsed_batch = (
        time.time()
        - batch_start
    )


    max_abs_cds_difference = int(
        df_batch_audit[
            "CDS Difference"
        ]
        .abs()
        .max()
    )


    min_observed_coverage = float(
        df_batch_audit[
            "Observed PLFam Ratio"
        ]
        .min()
    )


    print(
        f"[{batch_number:03d}/{len(batches):03d}] "
        f"PASS | "
        f"{len(batch_ids):2d} genomes | "
        f"{len(df_batch_membership):,} pairs | "
        f"min_cov={min_observed_coverage:.4f} | "
        f"max_CDS_delta={max_abs_cds_difference} | "
        f"{elapsed_batch:.1f}s"
    )


# =========================================================
# 8. Combine all batch audits
# =========================================================

audit_frames = []


for batch_number in range(
    1,
    len(batches) + 1
):

    path = os.path.join(
        PLFAM_BATCH_AUDIT_DIR,
        f"plfam_membership_audit_{batch_number:03d}.csv"
    )


    assert os.path.exists(
        path
    )


    frame = pd.read_csv(
        path,
        dtype={
            "Genome ID": str
        }
    )


    # -----------------------------------------------------
    # Harmonize old audits from batches 1–38
    # -----------------------------------------------------

    rename_map = {
        "Expected CDS":
            "Metadata CDS",

        "Expected PLFam CDS":
            "Metadata PLFam CDS"
    }


    frame = frame.rename(
        columns=rename_map
    )


    if "CDS Difference" not in frame.columns:

        frame[
            "CDS Difference"
        ] = (
            pd.to_numeric(
                frame[
                    "Returned CDS"
                ],
                errors="coerce"
            )
            -
            pd.to_numeric(
                frame[
                    "Metadata CDS"
                ],
                errors="coerce"
            )
        )


    if "PLFam CDS Difference" not in frame.columns:

        frame[
            "PLFam CDS Difference"
        ] = (
            pd.to_numeric(
                frame[
                    "Returned PLFam CDS"
                ],
                errors="coerce"
            )
            -
            pd.to_numeric(
                frame[
                    "Metadata PLFam CDS"
                ],
                errors="coerce"
            )
        )


    if "Metadata PLFam Ratio" not in frame.columns:

        metadata_ratio_map = (
            expected_counts
            .set_index(
                "Genome ID"
            )[
                "PLFam Ratio"
            ]
        )


        frame[
            "Metadata PLFam Ratio"
        ] = (
            frame[
                "Genome ID"
            ]
            .astype(str)
            .map(
                metadata_ratio_map
            )
        )


    if "Observed PLFam Ratio" not in frame.columns:

        frame[
            "Observed PLFam Ratio"
        ] = (
            pd.to_numeric(
                frame[
                    "Returned PLFam CDS"
                ],
                errors="coerce"
            )
            /
            pd.to_numeric(
                frame[
                    "Returned CDS"
                ],
                errors="coerce"
            )
        )


    if "Ratio Difference" not in frame.columns:

        frame[
            "Ratio Difference"
        ] = (
            frame[
                "Observed PLFam Ratio"
            ]
            -
            frame[
                "Metadata PLFam Ratio"
            ]
        )


    audit_frames.append(
        frame
    )


df_plfam_retrieval_audit = pd.concat(
    audit_frames,
    ignore_index=True
)


df_plfam_retrieval_audit[
    "Genome ID"
] = (
    df_plfam_retrieval_audit[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


# =========================================================
# 9. Final integrity checks
# =========================================================

assert len(
    df_plfam_retrieval_audit
) == 4227


assert (
    df_plfam_retrieval_audit[
        "Genome ID"
    ]
    .nunique()
    == 4227
)


assert (
    df_plfam_retrieval_audit[
        "Status"
    ]
    .eq("PASS")
    .all()
)


assert (
    pd.to_numeric(
        df_plfam_retrieval_audit[
            "Observed PLFam Ratio"
        ],
        errors="coerce"
    )
    .ge(
        MIN_OBSERVED_PLFAM_COVERAGE
    )
    .all()
)


# =========================================================
# 10. Save master audit
# =========================================================

df_plfam_retrieval_audit.to_csv(
    PLFAM_MASTER_AUDIT_PATH,
    index=False
)


# =========================================================
# 11. Final summary
# =========================================================

elapsed_all = (
    time.time()
    - start_all
)


total_plfam_cds = int(
    pd.to_numeric(
        df_plfam_retrieval_audit[
            "Returned PLFam CDS"
        ],
        errors="coerce"
    )
    .sum()
)


max_cds_delta = int(
    pd.to_numeric(
        df_plfam_retrieval_audit[
            "CDS Difference"
        ],
        errors="coerce"
    )
    .abs()
    .max()
)


n_cds_mismatch = int(
    (
        pd.to_numeric(
            df_plfam_retrieval_audit[
                "CDS Difference"
            ],
            errors="coerce"
        )
        != 0
    )
    .sum()
)


min_actual_coverage = float(
    pd.to_numeric(
        df_plfam_retrieval_audit[
            "Observed PLFam Ratio"
        ],
        errors="coerce"
    )
    .min()
)


print()
print("=" * 70)
print("PLFAM MEMBERSHIP RETRIEVAL COMPLETE")
print("=" * 70)

print(
    f"Pangenome genomes          : "
    f"{len(df_plfam_retrieval_audit):,}"
)

print(
    f"Successful genomes         : "
    f"{df_plfam_retrieval_audit['Status'].eq('PASS').sum():,}"
)

print(
    f"Completed batches          : "
    f"{completed_batches}/{len(batches)}"
)

print(
    f"New batches this run       : "
    f"{new_batches}"
)

print(
    f"Total PLFam-assigned CDS   : "
    f"{total_plfam_cds:,}"
)

print(
    f"Genomes with CDS metadata/API difference: "
    f"{n_cds_mismatch:,}"
)

print(
    f"Maximum absolute CDS difference          : "
    f"{max_cds_delta:,}"
)

print(
    f"Minimum observed PLFam coverage          : "
    f"{min_actual_coverage:.4%}"
)

print(
    f"Runtime this run           : "
    f"{elapsed_all / 60:.1f} min"
)

print()


print(
    "Unique PLFams per genome:"
)

display(
    pd.to_numeric(
        df_plfam_retrieval_audit[
            "Unique PLFams"
        ],
        errors="coerce"
    )
    .describe()
    .to_frame(
        name="Unique PLFams"
    )
)


print()
print(
    "Largest metadata/API CDS differences:"
)

display(
    df_plfam_retrieval_audit[
        [
            "Genome ID",
            "Metadata CDS",
            "Returned CDS",
            "CDS Difference",
            "Metadata PLFam CDS",
            "Returned PLFam CDS",
            "Observed PLFam Ratio"
        ]
    ]
    .assign(
        AbsDifference=lambda d:
        pd.to_numeric(
            d[
                "CDS Difference"
            ],
            errors="coerce"
        ).abs()
    )
    .sort_values(
        "AbsDifference",
        ascending=False
    )
    .head(20)
)


print()
print(
    "Master audit saved to:"
)

print(
    PLFAM_MASTER_AUDIT_PATH
)

print()

print(
    "PASS: PLFam membership was retrieved "
    "and validated for all 4,227 pangenome genomes."
)

PLFAM MEMBERSHIP RETRIEVAL — RESUME
Pangenome genomes       : 4,227
Batch size              : 50
Total batches           : 85
Concurrent requests     : 4
Minimum observed coverage: 95%

[001/085] SKIP — already complete
[002/085] SKIP — already complete
[003/085] SKIP — already complete
[004/085] SKIP — already complete
[005/085] SKIP — already complete
[006/085] SKIP — already complete
[007/085] SKIP — already complete
[008/085] SKIP — already complete
[009/085] SKIP — already complete
[010/085] SKIP — already complete
[011/085] SKIP — already complete
[012/085] SKIP — already complete
[013/085] SKIP — already complete
[014/085] SKIP — already complete
[015/085] SKIP — already complete
[016/085] SKIP — already complete
[017/085] SKIP — already complete
[018/085] SKIP — already complete
[019/085] SKIP — already complete
[020/085] SKIP — already complete
[021/085] SKIP — already complete
[022/085] SKIP — already complete
[023/085] SKIP — already complete
[024/085] SKIP — already complet

,Unique PLFams
count,4227.000000
mean,5311.291696
std,208.133349
min,1892.000000
25%,5191.000000
50%,5312.000000
75%,5433.000000
max,6569.000000



Largest metadata/API CDS differences:


,Genome ID,Metadata CDS,Returned CDS,CDS Difference,Metadata PLFam CDS,Returned PLFam CDS,Observed PLFam Ratio,AbsDifference
1945,573.1721,5921,5909,-12,5880,5880,0.995092,12
2000,573.2017,5688,5679,-9,5679,5679,1.000000,9
2008,573.2026,5714,5705,-9,5701,5701,0.999299,9
2002,573.2019,5736,5729,-7,5726,5726,0.999476,7
1950,573.1727,5938,5932,-6,5907,5907,0.995786,6
2006,573.2023,5760,5755,-5,5751,5751,0.999305,5
1984,573.2006,5677,5672,-5,5669,5669,0.999471,5
2004,573.2021,5722,5717,-5,5715,5715,0.999650,5
2001,573.2018,5609,5605,-4,5601,5601,0.999286,4
1976,573.1999,5706,5702,-4,5699,5699,0.999474,4



Master audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/pangenome/plfam_membership_retrieval_audit.csv

PASS: PLFam membership was retrieved and validated for all 4,227 pangenome genomes.


# Cell 9 — Audit the Full PLFam Prevalence Spectrum

In [11]:
import os
import glob

from collections import Counter

import numpy as np
import pandas as pd


# =========================================================
# 1. Persistent paths
# =========================================================

PANGENOME_MANIFEST_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_analysis_manifest.csv"
)

PLFAM_BATCH_DIR = os.path.join(
    PANGENOME_DIR,
    "plfam_membership_batches"
)

PLFAM_MASTER_AUDIT_PATH = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "plfam_membership_retrieval_audit.csv"
)

PLFAM_PREVALENCE_PATH = os.path.join(
    PANGENOME_DIR,
    "plfam_full_prevalence_audit.csv"
)

PLFAM_SPECTRUM_PATH = os.path.join(
    PANGENOME_DIR,
    "plfam_frequency_spectrum_summary.csv"
)


# =========================================================
# 2. Reload study manifest
# =========================================================

df_manifest_09 = pd.read_csv(
    PANGENOME_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_manifest_09[
    "Genome ID"
] = (
    df_manifest_09[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


expected_genome_ids = set(
    df_manifest_09[
        "Genome ID"
    ]
)


assert len(
    expected_genome_ids
) == 4227


# =========================================================
# 3. Reload Cell 8 master audit
# =========================================================

df_retrieval_audit_09 = pd.read_csv(
    PLFAM_MASTER_AUDIT_PATH,
    dtype={
        "Genome ID": str
    }
)


df_retrieval_audit_09[
    "Genome ID"
] = (
    df_retrieval_audit_09[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_retrieval_audit_09
) == 4227


assert (
    df_retrieval_audit_09[
        "Status"
    ]
    .eq("PASS")
    .all()
)


expected_total_plfam_cds = int(
    pd.to_numeric(
        df_retrieval_audit_09[
            "Returned PLFam CDS"
        ],
        errors="raise"
    )
    .sum()
)


expected_unique_per_genome = (
    df_retrieval_audit_09
    .set_index(
        "Genome ID"
    )[
        "Unique PLFams"
    ]
    .astype(int)
)


# =========================================================
# 4. Locate all saved membership batches
# =========================================================

batch_files = sorted(
    glob.glob(
        os.path.join(
            PLFAM_BATCH_DIR,
            "plfam_membership_batch_*.csv.gz"
        )
    )
)


assert len(
    batch_files
) == 85, (
    f"Expected 85 membership batch files, "
    f"found {len(batch_files)}."
)


print("=" * 70)
print("FULL PLFAM PREVALENCE AUDIT")
print("=" * 70)

print(
    f"Pangenome genomes      : "
    f"{len(expected_genome_ids):,}"
)

print(
    f"Membership batches     : "
    f"{len(batch_files):,}"
)

print(
    f"Expected PLFam CDS     : "
    f"{expected_total_plfam_cds:,}"
)

print()


# =========================================================
# 5. Streaming aggregation
# =========================================================

genome_presence_counter = Counter()

family_copy_counter = Counter()

genome_family_counter = Counter()

seen_genomes = set()

total_pair_rows = 0

total_copy_number = 0


for batch_index, batch_path in enumerate(
    batch_files,
    start=1
):

    df_batch = pd.read_csv(
        batch_path,
        dtype={
            "Genome ID": str,
            "PLFam ID": str,
            "Copy Number": int
        }
    )


    df_batch[
        "Genome ID"
    ] = (
        df_batch[
            "Genome ID"
        ]
        .astype(str)
        .str.strip()
    )


    df_batch[
        "PLFam ID"
    ] = (
        df_batch[
            "PLFam ID"
        ]
        .astype(str)
        .str.strip()
    )


    # -----------------------------------------------------
    # Batch integrity
    # -----------------------------------------------------

    assert (
        df_batch[
            "Genome ID"
        ]
        .notna()
        .all()
    )


    assert (
        df_batch[
            "PLFam ID"
        ]
        .notna()
        .all()
    )


    assert (
        df_batch[
            "Copy Number"
        ]
        .ge(1)
        .all()
    )


    duplicate_pairs = int(
        df_batch
        .duplicated(
            subset=[
                "Genome ID",
                "PLFam ID"
            ]
        )
        .sum()
    )


    assert duplicate_pairs == 0, (
        f"Duplicate genome-family pairs detected "
        f"in batch {batch_index}."
    )


    # -----------------------------------------------------
    # Genome coverage
    # -----------------------------------------------------

    batch_genomes = set(
        df_batch[
            "Genome ID"
        ]
    )


    assert batch_genomes.issubset(
        expected_genome_ids
    )


    seen_genomes.update(
        batch_genomes
    )


    # -----------------------------------------------------
    # One row = one genome-family presence
    # -----------------------------------------------------

    family_presence = (
        df_batch[
            "PLFam ID"
        ]
        .value_counts()
    )


    genome_presence_counter.update(
        family_presence.to_dict()
    )


    # -----------------------------------------------------
    # Total CDS copies per family
    # -----------------------------------------------------

    family_copies = (
        df_batch
        .groupby(
            "PLFam ID",
            sort=False
        )[
            "Copy Number"
        ]
        .sum()
    )


    family_copy_counter.update(
        family_copies.to_dict()
    )


    # -----------------------------------------------------
    # Unique families per genome
    # -----------------------------------------------------

    per_genome_pairs = (
        df_batch[
            "Genome ID"
        ]
        .value_counts()
    )


    genome_family_counter.update(
        per_genome_pairs.to_dict()
    )


    total_pair_rows += len(
        df_batch
    )


    total_copy_number += int(
        df_batch[
            "Copy Number"
        ]
        .sum()
    )


    if (
        batch_index % 10 == 0
        or batch_index == len(
            batch_files
        )
    ):

        print(
            f"[{batch_index:02d}/{len(batch_files):02d}] "
            f"pairs={total_pair_rows:,} | "
            f"copies={total_copy_number:,} | "
            f"families={len(genome_presence_counter):,}"
        )


# =========================================================
# 6. Global integrity checks
# =========================================================

assert seen_genomes == expected_genome_ids, (
    "Membership batches do not cover the exact "
    "4,227-genome pangenome cohort."
)


assert total_copy_number == expected_total_plfam_cds, (
    f"PLFam CDS reconstruction mismatch: "
    f"expected={expected_total_plfam_cds:,}, "
    f"observed={total_copy_number:,}."
)


# =========================================================
# 7. Verify per-genome family counts
# =========================================================

reconstructed_unique_per_genome = pd.Series(
    genome_family_counter,
    name="Reconstructed Unique PLFams"
)


reconstructed_unique_per_genome.index = (
    reconstructed_unique_per_genome.index
    .astype(str)
)


reconstructed_unique_per_genome = (
    reconstructed_unique_per_genome
    .reindex(
        expected_unique_per_genome.index
    )
    .astype(int)
)


per_genome_difference = (
    reconstructed_unique_per_genome
    - expected_unique_per_genome
)


assert (
    per_genome_difference
    .eq(0)
    .all()
), (
    "Saved membership batches do not reproduce "
    "Cell 8 Unique PLFam counts."
)


# =========================================================
# 8. Build global PLFam prevalence table
# =========================================================

all_plfam_ids = sorted(
    genome_presence_counter.keys()
)


df_plfam_prevalence = pd.DataFrame(
    {
        "PLFam ID":
            all_plfam_ids
    }
)


df_plfam_prevalence[
    "Genomes Present"
] = (
    df_plfam_prevalence[
        "PLFam ID"
    ]
    .map(
        genome_presence_counter
    )
    .astype(int)
)


df_plfam_prevalence[
    "Prevalence"
] = (
    df_plfam_prevalence[
        "Genomes Present"
    ]
    / len(
        expected_genome_ids
    )
)


df_plfam_prevalence[
    "Total Gene Copies"
] = (
    df_plfam_prevalence[
        "PLFam ID"
    ]
    .map(
        family_copy_counter
    )
    .astype(int)
)


df_plfam_prevalence[
    "Extra Copies"
] = (
    df_plfam_prevalence[
        "Total Gene Copies"
    ]
    -
    df_plfam_prevalence[
        "Genomes Present"
    ]
)


df_plfam_prevalence = (
    df_plfam_prevalence
    .sort_values(
        [
            "Genomes Present",
            "PLFam ID"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)


# =========================================================
# 9. Frequency-spectrum summary
# =========================================================

n_genomes = len(
    expected_genome_ids
)


def n_families_at_least(
    minimum_genomes
):

    return int(
        (
            df_plfam_prevalence[
                "Genomes Present"
            ]
            >= minimum_genomes
        )
        .sum()
    )


def n_variable_families_at_least(
    minimum_genomes
):

    return int(
        (
            (
                df_plfam_prevalence[
                    "Genomes Present"
                ]
                >= minimum_genomes
            )
            &
            (
                df_plfam_prevalence[
                    "Genomes Present"
                ]
                < n_genomes
            )
        )
        .sum()
    )


threshold_1pct = int(
    np.ceil(
        n_genomes
        * 0.01
    )
)


spectrum_records = [
    {
        "Category":
            "Total observed PLFams",

        "Families":
            len(
                df_plfam_prevalence
            )
    },
    {
        "Category":
            "Singletons (1 genome)",

        "Families":
            int(
                (
                    df_plfam_prevalence[
                        "Genomes Present"
                    ]
                    == 1
                )
                .sum()
            )
    },
    {
        "Category":
            "Present in >=2 genomes",

        "Families":
            n_families_at_least(
                2
            )
    },
    {
        "Category":
            "Present in >=5 genomes",

        "Families":
            n_families_at_least(
                5
            )
    },
    {
        "Category":
            "Present in >=10 genomes",

        "Families":
            n_families_at_least(
                10
            )
    },
    {
        "Category":
            f"Present in >=1% genomes (>= {threshold_1pct})",

        "Families":
            n_families_at_least(
                threshold_1pct
            )
    },
    {
        "Category":
            "Present in all 4,227 genomes",

        "Families":
            int(
                (
                    df_plfam_prevalence[
                        "Genomes Present"
                    ]
                    == n_genomes
                )
                .sum()
            )
    },
    {
        "Category":
            "Variable and present in >=5 genomes",

        "Families":
            n_variable_families_at_least(
                5
            )
    }
]


df_plfam_spectrum = pd.DataFrame(
    spectrum_records
)


# =========================================================
# 10. Save outputs
# =========================================================

df_plfam_prevalence.to_csv(
    PLFAM_PREVALENCE_PATH,
    index=False
)


df_plfam_spectrum.to_csv(
    PLFAM_SPECTRUM_PATH,
    index=False
)


# =========================================================
# 11. Final report
# =========================================================

print()
print("=" * 70)
print("PLFAM PREVALENCE SPECTRUM")
print("=" * 70)

print(
    f"Genome-family presence pairs : "
    f"{total_pair_rows:,}"
)

print(
    f"Reconstructed PLFam CDS      : "
    f"{total_copy_number:,}"
)

print(
    f"Observed unique PLFams        : "
    f"{len(df_plfam_prevalence):,}"
)

print()


display(
    df_plfam_spectrum
)


print()
print(
    "PLFam prevalence distribution:"
)

display(
    df_plfam_prevalence[
        "Genomes Present"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .to_frame(
        name="Genomes Present"
    )
)


print()
print(
    "Most prevalent PLFams:"
)

display(
    df_plfam_prevalence
    .head(20)
)


print()
print(
    "Rare PLFams:"
)

display(
    df_plfam_prevalence
    .tail(20)
)


print()
print(
    "Prevalence audit saved to:"
)

print(
    PLFAM_PREVALENCE_PATH
)

print()

print(
    "PASS: The complete PLFam prevalence spectrum "
    "was reconstructed without using phenotype labels."
)

FULL PLFAM PREVALENCE AUDIT
Pangenome genomes      : 4,227
Membership batches     : 85
Expected PLFam CDS     : 22,908,951

[10/85] pairs=2,686,824 | copies=2,750,291 | families=23,523
[20/85] pairs=5,374,956 | copies=5,485,852 | families=29,215
[30/85] pairs=8,072,749 | copies=8,235,413 | families=31,577
[40/85] pairs=10,769,847 | copies=10,987,791 | families=37,268
[50/85] pairs=13,408,415 | copies=13,693,678 | families=40,257
[60/85] pairs=16,060,239 | copies=16,403,240 | families=42,679
[70/85] pairs=18,667,283 | copies=19,056,344 | families=43,514
[80/85] pairs=21,217,286 | copies=21,649,229 | families=44,616
[85/85] pairs=22,450,830 | copies=22,908,951 | families=45,492

PLFAM PREVALENCE SPECTRUM
Genome-family presence pairs : 22,450,830
Reconstructed PLFam CDS      : 22,908,951
Observed unique PLFams        : 45,492



,Category,Families
0,Total observed PLFams,45492
1,Singletons (1 genome),11209
2,Present in >=2 genomes,34283
3,Present in >=5 genomes,25222
4,Present in >=10 genomes,20267
5,Present in >=1% genomes (>= 43),12431
6,"Present in all 4,227 genomes",594
7,Variable and present in >=5 genomes,24628



PLFam prevalence distribution:


,Genomes Present
count,45492.000000
mean,493.511606
std,1256.039657
min,1.000000
25%,2.000000
50%,6.000000
75%,57.000000
90%,2993.700000
95%,4224.000000
99%,4227.000000



Most prevalent PLFams:


,PLFam ID,Genomes Present,Prevalence,Total Gene Copies,Extra Copies
0,PLF_570_00000010,4227,1.0,4228,1
1,PLF_570_00000014,4227,1.0,8526,4299
2,PLF_570_00000017,4227,1.0,8450,4223
3,PLF_570_00000025,4227,1.0,4510,283
4,PLF_570_00000087,4227,1.0,4429,202
5,PLF_570_00000091,4227,1.0,4231,4
6,PLF_570_00000100,4227,1.0,4463,236
7,PLF_570_00000130,4227,1.0,4245,18
8,PLF_570_00000137,4227,1.0,4234,7
9,PLF_570_00000144,4227,1.0,4231,4



Rare PLFams:


,PLFam ID,Genomes Present,Prevalence,Total Gene Copies,Extra Copies
45472,PLF_570_00131397,1,0.000237,1,0
45473,PLF_570_00131410,1,0.000237,1,0
45474,PLF_570_00131425,1,0.000237,1,0
45475,PLF_570_00131428,1,0.000237,1,0
45476,PLF_570_00131440,1,0.000237,1,0
45477,PLF_570_00131446,1,0.000237,2,1
45478,PLF_570_00131451,1,0.000237,1,0
45479,PLF_570_00131455,1,0.000237,1,0
45480,PLF_570_00131460,1,0.000237,1,0
45481,PLF_570_00131469,1,0.000237,1,0



Prevalence audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/plfam_full_prevalence_audit.csv

PASS: The complete PLFam prevalence spectrum was reconstructed without using phenotype labels.


# Cell 10 — Construct the Filtered Binary PLFam Pangenome Matrix

In [12]:
import os
import glob

import numpy as np
import pandas as pd

from scipy import sparse


# =========================================================
# 1. Persistent paths
# =========================================================

PANGENOME_MANIFEST_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_analysis_manifest.csv"
)

PLFAM_PREVALENCE_PATH = os.path.join(
    PANGENOME_DIR,
    "plfam_full_prevalence_audit.csv"
)

PLFAM_BATCH_DIR = os.path.join(
    PANGENOME_DIR,
    "plfam_membership_batches"
)


X_PANGENOME_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_plfam_binary.npz"
)

PANGENOME_ROW_INDEX_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_row_index.csv"
)

PANGENOME_FEATURE_INDEX_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_feature_index.csv"
)

PLFAM_FILTER_AUDIT_PATH = os.path.join(
    PANGENOME_DIR,
    "plfam_feature_filter_audit.csv"
)

PANGENOME_MATRIX_SUMMARY_PATH = os.path.join(
    PANGENOME_CHECKPOINT_DIR,
    "pangenome_matrix_build_summary.csv"
)


# =========================================================
# 2. Label-independent filtering rule
# =========================================================

MIN_MINOR_STATE_COUNT = 5


# =========================================================
# 3. Reload pangenome cohort
# =========================================================

df_manifest_10 = pd.read_csv(
    PANGENOME_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_manifest_10[
    "Genome ID"
] = (
    df_manifest_10[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


assert len(
    df_manifest_10
) == 4227


assert (
    df_manifest_10[
        "Genome ID"
    ]
    .nunique()
    == 4227
)


study_ids = (
    df_manifest_10[
        "Genome ID"
    ]
    .tolist()
)


n_genomes = len(
    study_ids
)


# =========================================================
# 4. Reload full PLFam prevalence spectrum
# =========================================================

df_prevalence_10 = pd.read_csv(
    PLFAM_PREVALENCE_PATH,
    dtype={
        "PLFam ID": str
    }
)


assert len(
    df_prevalence_10
) == 45492


df_prevalence_10[
    "Genomes Present"
] = pd.to_numeric(
    df_prevalence_10[
        "Genomes Present"
    ],
    errors="raise"
).astype(int)


# =========================================================
# 5. Calculate absence and minor-state count
# =========================================================

df_prevalence_10[
    "Genomes Absent"
] = (
    n_genomes
    -
    df_prevalence_10[
        "Genomes Present"
    ]
)


df_prevalence_10[
    "Minor State Count"
] = (
    df_prevalence_10[
        [
            "Genomes Present",
            "Genomes Absent"
        ]
    ]
    .min(
        axis=1
    )
    .astype(int)
)


# =========================================================
# 6. Apply phenotype-independent filter
# =========================================================

df_prevalence_10[
    "Retain Feature"
] = (
    df_prevalence_10[
        "Minor State Count"
    ]
    .ge(
        MIN_MINOR_STATE_COUNT
    )
)


def classify_filter_reason(row):

    if row[
        "Retain Feature"
    ]:

        return "RETAIN"


    if (
        row[
            "Genomes Present"
        ]
        < MIN_MINOR_STATE_COUNT
    ):

        return "TOO_RARE"


    if (
        row[
            "Genomes Absent"
        ]
        < MIN_MINOR_STATE_COUNT
    ):

        return "NEAR_CONSTANT"


    return "OTHER"


df_prevalence_10[
    "Filter Decision"
] = (
    df_prevalence_10
    .apply(
        classify_filter_reason,
        axis=1
    )
)


# =========================================================
# 7. Final feature set
# =========================================================

df_selected_features = (
    df_prevalence_10[
        df_prevalence_10[
            "Retain Feature"
        ]
    ]
    .copy()
)


df_selected_features = (
    df_selected_features
    .sort_values(
        "PLFam ID"
    )
    .reset_index(
        drop=True
    )
)


df_selected_features[
    "Feature Index"
] = np.arange(
    len(
        df_selected_features
    ),
    dtype=np.int32
)


selected_feature_ids = (
    df_selected_features[
        "PLFam ID"
    ]
    .tolist()
)


selected_feature_set = set(
    selected_feature_ids
)


feature_to_index = dict(
    zip(
        df_selected_features[
            "PLFam ID"
        ],
        df_selected_features[
            "Feature Index"
        ]
    )
)


n_features = len(
    df_selected_features
)


# =========================================================
# 8. Filter summary before matrix construction
# =========================================================

n_too_rare = int(
    df_prevalence_10[
        "Filter Decision"
    ]
    .eq(
        "TOO_RARE"
    )
    .sum()
)


n_near_constant = int(
    df_prevalence_10[
        "Filter Decision"
    ]
    .eq(
        "NEAR_CONSTANT"
    )
    .sum()
)


n_constant_present = int(
    df_prevalence_10[
        "Genomes Present"
    ]
    .eq(
        n_genomes
    )
    .sum()
)


print("=" * 70)
print("PLFAM FEATURE FILTER")
print("=" * 70)

print(
    f"Observed PLFams          : "
    f"{len(df_prevalence_10):,}"
)

print(
    f"Minor-state threshold    : "
    f"{MIN_MINOR_STATE_COUNT}"
)

print(
    f"Removed as too rare      : "
    f"{n_too_rare:,}"
)

print(
    f"Removed as near-constant : "
    f"{n_near_constant:,}"
)

print(
    f"Constant-present families: "
    f"{n_constant_present:,}"
)

print(
    f"Retained PLFam features  : "
    f"{n_features:,}"
)

print()


# =========================================================
# 9. Locate all membership batches
# =========================================================

batch_files = sorted(
    glob.glob(
        os.path.join(
            PLFAM_BATCH_DIR,
            "plfam_membership_batch_*.csv.gz"
        )
    )
)


assert len(
    batch_files
) == 85


# =========================================================
# 10. Construct sparse binary matrix batch-by-batch
# =========================================================

BATCH_SIZE = 50

batch_matrices = []

total_selected_pairs = 0


print("=" * 70)
print("SPARSE PANGENOME MATRIX CONSTRUCTION")
print("=" * 70)


for batch_number, batch_path in enumerate(
    batch_files,
    start=1
):

    start_index = (
        (batch_number - 1)
        * BATCH_SIZE
    )


    end_index = min(
        start_index
        + BATCH_SIZE,
        n_genomes
    )


    batch_genome_ids = (
        study_ids[
            start_index:end_index
        ]
    )


    local_row_map = {
        genome_id:
            row_index

        for row_index, genome_id
        in enumerate(
            batch_genome_ids
        )
    }


    df_batch = pd.read_csv(
        batch_path,
        dtype={
            "Genome ID": str,
            "PLFam ID": str,
            "Copy Number": int
        }
    )


    df_batch[
        "Genome ID"
    ] = (
        df_batch[
            "Genome ID"
        ]
        .astype(str)
        .str.strip()
    )


    df_batch[
        "PLFam ID"
    ] = (
        df_batch[
            "PLFam ID"
        ]
        .astype(str)
        .str.strip()
    )


    # -----------------------------------------------------
    # Verify exact batch cohort
    # -----------------------------------------------------

    observed_batch_ids = set(
        df_batch[
            "Genome ID"
        ]
    )


    assert observed_batch_ids == set(
        batch_genome_ids
    ), (
        f"Genome mismatch in batch "
        f"{batch_number}."
    )


    # -----------------------------------------------------
    # Retain selected PLFam features
    # -----------------------------------------------------

    df_selected_batch = (
        df_batch[
            df_batch[
                "PLFam ID"
            ]
            .isin(
                selected_feature_set
            )
        ][
            [
                "Genome ID",
                "PLFam ID"
            ]
        ]
        .copy()
    )


    # Each membership batch already contains one
    # row per genome-family combination.
    assert not df_selected_batch.duplicated(
        subset=[
            "Genome ID",
            "PLFam ID"
        ]
    ).any()


    rows = (
        df_selected_batch[
            "Genome ID"
        ]
        .map(
            local_row_map
        )
        .to_numpy(
            dtype=np.int32
        )
    )


    cols = (
        df_selected_batch[
            "PLFam ID"
        ]
        .map(
            feature_to_index
        )
        .to_numpy(
            dtype=np.int32
        )
    )


    data = np.ones(
        len(
            df_selected_batch
        ),
        dtype=np.uint8
    )


    X_batch = sparse.csr_matrix(
        (
            data,
            (
                rows,
                cols
            )
        ),
        shape=(
            len(
                batch_genome_ids
            ),
            n_features
        ),
        dtype=np.uint8
    )


    batch_matrices.append(
        X_batch
    )


    total_selected_pairs += (
        X_batch.nnz
    )


    if (
        batch_number % 10 == 0
        or batch_number == len(
            batch_files
        )
    ):

        print(
            f"[{batch_number:02d}/{len(batch_files):02d}] "
            f"selected pairs="
            f"{total_selected_pairs:,}"
        )


# =========================================================
# 11. Stack into final sparse matrix
# =========================================================

X_pangenome = sparse.vstack(
    batch_matrices,
    format="csr"
)


X_pangenome.eliminate_zeros()


# =========================================================
# 12. Matrix integrity checks
# =========================================================

assert X_pangenome.shape == (
    n_genomes,
    n_features
)


assert X_pangenome.dtype == np.uint8


assert (
    X_pangenome.data
    == 1
).all()


# ---------------------------------------------------------
# Feature prevalence must reproduce Cell 9
# ---------------------------------------------------------

matrix_feature_counts = np.asarray(
    X_pangenome.sum(
        axis=0
    )
).ravel().astype(int)


expected_feature_counts = (
    df_selected_features[
        "Genomes Present"
    ]
    .to_numpy(
        dtype=int
    )
)


assert np.array_equal(
    matrix_feature_counts,
    expected_feature_counts
), (
    "Sparse matrix feature prevalence does not "
    "match the Cell 9 prevalence audit."
)


# =========================================================
# 13. Row-level feature counts
# =========================================================

features_per_genome = np.asarray(
    X_pangenome.sum(
        axis=1
    )
).ravel().astype(int)


zero_feature_genomes = int(
    (
        features_per_genome
        == 0
    )
    .sum()
)


# =========================================================
# 14. Matrix density
# =========================================================

total_matrix_cells = (
    X_pangenome.shape[0]
    * X_pangenome.shape[1]
)


matrix_density = (
    X_pangenome.nnz
    / total_matrix_cells
)


matrix_sparsity = (
    1
    - matrix_density
)


# =========================================================
# 15. Save row index
# =========================================================

df_row_index = pd.DataFrame(
    {
        "Row Index":
            np.arange(
                n_genomes,
                dtype=np.int32
            ),

        "Genome ID":
            study_ids
    }
)


# =========================================================
# 16. Save feature index
# =========================================================

df_feature_index = (
    df_selected_features[
        [
            "Feature Index",
            "PLFam ID",
            "Genomes Present",
            "Genomes Absent",
            "Minor State Count",
            "Prevalence",
            "Total Gene Copies",
            "Extra Copies"
        ]
    ]
    .copy()
)


# =========================================================
# 17. Save filter audit
# =========================================================

df_prevalence_10.to_csv(
    PLFAM_FILTER_AUDIT_PATH,
    index=False
)


# =========================================================
# 18. Save sparse matrix + indices
# =========================================================

sparse.save_npz(
    X_PANGENOME_PATH,
    X_pangenome,
    compressed=True
)


df_row_index.to_csv(
    PANGENOME_ROW_INDEX_PATH,
    index=False
)


df_feature_index.to_csv(
    PANGENOME_FEATURE_INDEX_PATH,
    index=False
)


# =========================================================
# 19. Save build summary
# =========================================================

df_matrix_summary = pd.DataFrame(
    [
        {
            "Representation":
                "BV-BRC PLFam binary presence/absence",

            "Study Genomes":
                n_genomes,

            "Observed PLFams":
                len(
                    df_prevalence_10
                ),

            "Minor State Threshold":
                MIN_MINOR_STATE_COUNT,

            "Too-Rare Features Removed":
                n_too_rare,

            "Near-Constant Features Removed":
                n_near_constant,

            "Final Features":
                n_features,

            "Matrix Nonzero Entries":
                int(
                    X_pangenome.nnz
                ),

            "Matrix Density":
                matrix_density,

            "Zero-Feature Genomes":
                zero_feature_genomes,

            "Phenotype Used for Filtering":
                False
        }
    ]
)


df_matrix_summary.to_csv(
    PANGENOME_MATRIX_SUMMARY_PATH,
    index=False
)


# =========================================================
# 20. Final report
# =========================================================

print()
print("=" * 70)
print("FINAL BINARY PLFAM PANGENOME MATRIX")
print("=" * 70)

print(
    f"Study genomes             : "
    f"{X_pangenome.shape[0]:,}"
)

print(
    f"Observed PLFams           : "
    f"{len(df_prevalence_10):,}"
)

print(
    f"Filtered PLFam features   : "
    f"{X_pangenome.shape[1]:,}"
)

print(
    f"Matrix dimensions         : "
    f"{X_pangenome.shape[0]:,} x "
    f"{X_pangenome.shape[1]:,}"
)

print(
    f"Nonzero entries           : "
    f"{X_pangenome.nnz:,}"
)

print(
    f"Matrix density            : "
    f"{matrix_density:.4%}"
)

print(
    f"Matrix sparsity           : "
    f"{matrix_sparsity:.4%}"
)

print(
    f"Zero-feature genomes      : "
    f"{zero_feature_genomes:,}"
)

print()


print(
    "Retained features per genome:"
)

display(
    pd.Series(
        features_per_genome
    )
    .describe()
    .to_frame(
        name="Feature Count"
    )
)


print()
print(
    "Filter-decision counts:"
)

print(
    df_prevalence_10[
        "Filter Decision"
    ]
    .value_counts()
)


print()
print(
    "Sparse matrix saved to:"
)

print(
    X_PANGENOME_PATH
)

print()

print(
    "Row index saved to:"
)

print(
    PANGENOME_ROW_INDEX_PATH
)

print()

print(
    "Feature index saved to:"
)

print(
    PANGENOME_FEATURE_INDEX_PATH
)

print()

print(
    "PASS: The phenotype-independent binary "
    "PLFam pangenome matrix was constructed successfully."
)

PLFAM FEATURE FILTER
Observed PLFams          : 45,492
Minor-state threshold    : 5
Removed as too rare      : 20,270
Removed as near-constant : 2,537
Constant-present families: 594
Retained PLFam features  : 22,685

SPARSE PANGENOME MATRIX CONSTRUCTION
[10/85] selected pairs=1,413,314
[20/85] selected pairs=2,828,685
[30/85] selected pairs=4,255,655
[40/85] selected pairs=5,677,894
[50/85] selected pairs=7,043,271
[60/85] selected pairs=8,423,880
[70/85] selected pairs=9,760,888
[80/85] selected pairs=11,038,764
[85/85] selected pairs=11,694,824

FINAL BINARY PLFAM PANGENOME MATRIX
Study genomes             : 4,227
Observed PLFams           : 45,492
Filtered PLFam features   : 22,685
Matrix dimensions         : 4,227 x 22,685
Nonzero entries           : 11,694,824
Matrix density            : 12.1961%
Matrix sparsity           : 87.8039%
Zero-feature genomes      : 0

Retained features per genome:


,Feature Count
count,4227.000000
mean,2766.696002
std,202.462943
min,651.000000
25%,2650.000000
50%,2768.000000
75%,2889.000000
max,3955.000000



Filter-decision counts:
Filter Decision
RETAIN           22685
TOO_RARE         20270
NEAR_CONSTANT     2537
Name: count, dtype: int64

Sparse matrix saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/X_pangenome_plfam_binary.npz

Row index saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/X_pangenome_row_index.csv

Feature index saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/X_pangenome_feature_index.csv

PASS: The phenotype-independent binary PLFam pangenome matrix was constructed successfully.


# Cell 11 — Final Integrity Audit of the PLFam Pangenome Matrix

In [13]:
import os
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.stats import fisher_exact


# =========================================================
# 1. Persistent paths
# =========================================================

X_PANGENOME_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_plfam_binary.npz"
)

PANGENOME_ROW_INDEX_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_row_index.csv"
)

PANGENOME_FEATURE_INDEX_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_feature_index.csv"
)

PANGENOME_MANIFEST_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_analysis_manifest.csv"
)

PANGENOME_EXCLUDED_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_technical_exclusions.csv"
)

PANGENOME_FINAL_AUDIT_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_final_integrity_audit.csv"
)

PANGENOME_EXCLUSION_BIAS_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_exclusion_bias_audit.csv"
)


# =========================================================
# 2. Verify required outputs
# =========================================================

required_paths = [
    X_PANGENOME_PATH,
    PANGENOME_ROW_INDEX_PATH,
    PANGENOME_FEATURE_INDEX_PATH,
    PANGENOME_MANIFEST_PATH,
    PANGENOME_EXCLUDED_PATH
]


missing_paths = [
    path
    for path in required_paths
    if not os.path.exists(path)
]


assert not missing_paths, (
    "Missing File 05 outputs:\n"
    + "\n".join(missing_paths)
)


# =========================================================
# 3. Reload finalized sparse matrix
# =========================================================

X_pangenome_11 = sparse.load_npz(
    X_PANGENOME_PATH
).tocsr()


# =========================================================
# 4. Reload row / feature indices
# =========================================================

df_row_index_11 = pd.read_csv(
    PANGENOME_ROW_INDEX_PATH,
    dtype={
        "Genome ID": str
    }
)


df_feature_index_11 = pd.read_csv(
    PANGENOME_FEATURE_INDEX_PATH,
    dtype={
        "PLFam ID": str
    }
)


df_manifest_11 = pd.read_csv(
    PANGENOME_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_excluded_11 = pd.read_csv(
    PANGENOME_EXCLUDED_PATH,
    dtype={
        "Genome ID": str
    }
)


for df in [
    df_row_index_11,
    df_manifest_11,
    df_excluded_11
]:

    df["Genome ID"] = (
        df["Genome ID"]
        .astype(str)
        .str.strip()
    )


# =========================================================
# 5. Basic dimensions
# =========================================================

n_genomes, n_features = (
    X_pangenome_11.shape
)


assert n_genomes == 4227

assert n_features == 22685

assert len(
    df_row_index_11
) == n_genomes

assert len(
    df_feature_index_11
) == n_features


# =========================================================
# 6. Row-index integrity
# =========================================================

assert (
    df_row_index_11[
        "Row Index"
    ]
    .tolist()
    ==
    list(
        range(
            n_genomes
        )
    )
)


assert (
    df_row_index_11[
        "Genome ID"
    ]
    .nunique()
    == n_genomes
)


assert (
    df_manifest_11[
        "Genome ID"
    ]
    .nunique()
    == n_genomes
)


assert (
    df_row_index_11[
        "Genome ID"
    ]
    .tolist()
    ==
    df_manifest_11[
        "Genome ID"
    ]
    .tolist()
), (
    "Row index does not match the finalized "
    "pangenome cohort order."
)


# =========================================================
# 7. Feature-index integrity
# =========================================================

assert (
    df_feature_index_11[
        "Feature Index"
    ]
    .tolist()
    ==
    list(
        range(
            n_features
        )
    )
)


assert (
    df_feature_index_11[
        "PLFam ID"
    ]
    .nunique()
    == n_features
)


# =========================================================
# 8. Binary sparse encoding audit
# =========================================================

unique_nonzero_values = set(
    np.unique(
        X_pangenome_11.data
    )
)


binary_valid = (
    unique_nonzero_values
    == {1}
)


assert binary_valid, (
    f"Unexpected sparse matrix values: "
    f"{unique_nonzero_values}"
)


# =========================================================
# 9. Missing / empty row audit
# =========================================================

row_counts = np.asarray(
    X_pangenome_11.sum(
        axis=1
    )
).ravel().astype(int)


zero_feature_genomes = int(
    (
        row_counts
        == 0
    )
    .sum()
)


assert zero_feature_genomes == 0


# =========================================================
# 10. Feature prevalence reconstruction
# =========================================================

matrix_feature_counts = np.asarray(
    X_pangenome_11.sum(
        axis=0
    )
).ravel().astype(int)


expected_feature_counts = (
    df_feature_index_11[
        "Genomes Present"
    ]
    .astype(int)
    .to_numpy()
)


assert np.array_equal(
    matrix_feature_counts,
    expected_feature_counts
), (
    "Matrix prevalence does not match "
    "the saved feature index."
)


# =========================================================
# 11. Minor-state filter verification
# =========================================================

matrix_absence_counts = (
    n_genomes
    - matrix_feature_counts
)


matrix_minor_state = np.minimum(
    matrix_feature_counts,
    matrix_absence_counts
)


min_minor_state = int(
    matrix_minor_state.min()
)


assert min_minor_state >= 5


assert (
    matrix_feature_counts
    < n_genomes
).all(), (
    "Constant-present feature detected."
)


assert (
    matrix_feature_counts
    > 0
).all(), (
    "Constant-absent feature detected."
)


# =========================================================
# 12. Matrix density
# =========================================================

matrix_nnz = int(
    X_pangenome_11.nnz
)


matrix_density = (
    matrix_nnz
    /
    (
        n_genomes
        * n_features
    )
)


matrix_sparsity = (
    1
    - matrix_density
)


# =========================================================
# 13. Verify excluded cohort
# =========================================================

assert len(
    df_excluded_11
) == 6


assert set(
    df_excluded_11[
        "Genome ID"
    ]
).isdisjoint(
    set(
        df_row_index_11[
            "Genome ID"
        ]
    )
)


# =========================================================
# 14. Phenotype exclusion-bias audit
#     (audit only; not feature filtering)
# =========================================================

full_phenotype = (
    df_pangenome_input[
        [
            "Genome ID",
            "Resistant Phenotype"
        ]
    ]
    .copy()
)


full_phenotype[
    "Genome ID"
] = (
    full_phenotype[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


retained_ids = set(
    df_row_index_11[
        "Genome ID"
    ]
)


full_phenotype[
    "Retained"
] = (
    full_phenotype[
        "Genome ID"
    ]
    .isin(
        retained_ids
    )
)


# Rows:
# Resistant / Susceptible
# Columns:
# Excluded / Retained

r_excluded = int(
    (
        full_phenotype[
            "Resistant Phenotype"
        ]
        .eq("Resistant")
        &
        ~full_phenotype[
            "Retained"
        ]
    )
    .sum()
)


r_retained = int(
    (
        full_phenotype[
            "Resistant Phenotype"
        ]
        .eq("Resistant")
        &
        full_phenotype[
            "Retained"
        ]
    )
    .sum()
)


s_excluded = int(
    (
        full_phenotype[
            "Resistant Phenotype"
        ]
        .eq("Susceptible")
        &
        ~full_phenotype[
            "Retained"
        ]
    )
    .sum()
)


s_retained = int(
    (
        full_phenotype[
            "Resistant Phenotype"
        ]
        .eq("Susceptible")
        &
        full_phenotype[
            "Retained"
        ]
    )
    .sum()
)


odds_ratio, fisher_p = fisher_exact(
    [
        [
            r_excluded,
            r_retained
        ],
        [
            s_excluded,
            s_retained
        ]
    ],
    alternative="two-sided"
)


df_exclusion_bias = pd.DataFrame(
    [
        {
            "Resistant Excluded":
                r_excluded,

            "Resistant Retained":
                r_retained,

            "Susceptible Excluded":
                s_excluded,

            "Susceptible Retained":
                s_retained,

            "Fisher Odds Ratio":
                odds_ratio,

            "Fisher P Value":
                fisher_p,

            "Phenotype Used for Cohort Rule":
                False
        }
    ]
)


# =========================================================
# 15. Final audit table
# =========================================================

audit_records = [
    {
        "Metric":
            "Pangenome genomes",

        "Value":
            n_genomes
    },
    {
        "Metric":
            "PLFam features",

        "Value":
            n_features
    },
    {
        "Metric":
            "Matrix nonzero entries",

        "Value":
            matrix_nnz
    },
    {
        "Metric":
            "Matrix density",

        "Value":
            matrix_density
    },
    {
        "Metric":
            "Matrix sparsity",

        "Value":
            matrix_sparsity
    },
    {
        "Metric":
            "Zero-feature genomes",

        "Value":
            zero_feature_genomes
    },
    {
        "Metric":
            "Minimum minor-state count",

        "Value":
            min_minor_state
    },
    {
        "Metric":
            "Constant features",

        "Value":
            int(
                (
                    matrix_minor_state
                    == 0
                )
                .sum()
            )
    },
    {
        "Metric":
            "Technical cohort exclusions",

        "Value":
            len(
                df_excluded_11
            )
    },
    {
        "Metric":
            "Fisher exclusion-bias p-value",

        "Value":
            fisher_p
    }
]


df_pangenome_final_audit = pd.DataFrame(
    audit_records
)


# =========================================================
# 16. Save audits
# =========================================================

df_pangenome_final_audit.to_csv(
    PANGENOME_FINAL_AUDIT_PATH,
    index=False
)


df_exclusion_bias.to_csv(
    PANGENOME_EXCLUSION_BIAS_PATH,
    index=False
)


# =========================================================
# 17. Final report
# =========================================================

print("=" * 70)
print("FINAL PANGENOME MATRIX INTEGRITY AUDIT")
print("=" * 70)

print(
    f"Pangenome genomes        : "
    f"{n_genomes:,}"
)

print(
    f"PLFam features           : "
    f"{n_features:,}"
)

print(
    f"Matrix dimensions        : "
    f"{n_genomes:,} x "
    f"{n_features:,}"
)

print(
    f"Nonzero entries          : "
    f"{matrix_nnz:,}"
)

print(
    f"Binary sparse encoding   : "
    f"{binary_valid}"
)

print(
    f"Zero-feature genomes     : "
    f"{zero_feature_genomes:,}"
)

print(
    f"Minimum minor-state count: "
    f"{min_minor_state:,}"
)

print(
    f"Constant features        : "
    f"{(matrix_minor_state == 0).sum():,}"
)

print(
    f"Matrix density           : "
    f"{matrix_density:.4%}"
)

print(
    f"Matrix sparsity          : "
    f"{matrix_sparsity:.4%}"
)

print()


print(
    "Retained features per genome:"
)

display(
    pd.Series(
        row_counts
    )
    .describe()
    .to_frame(
        name="Feature Count"
    )
)


print()
print(
    "Technical exclusion bias audit:"
)

display(
    df_exclusion_bias
)


print()
print(
    "Final audit saved to:"
)

print(
    PANGENOME_FINAL_AUDIT_PATH
)

print()

print(
    "PASS: The finalized PLFam pangenome matrix "
    "passed the File 05 integrity audit."
)

FINAL PANGENOME MATRIX INTEGRITY AUDIT
Pangenome genomes        : 4,227
PLFam features           : 22,685
Matrix dimensions        : 4,227 x 22,685
Nonzero entries          : 11,694,824
Binary sparse encoding   : True
Zero-feature genomes     : 0
Minimum minor-state count: 5
Constant features        : 0
Matrix density           : 12.1961%
Matrix sparsity          : 87.8039%

Retained features per genome:


,Feature Count
count,4227.000000
mean,2766.696002
std,202.462943
min,651.000000
25%,2650.000000
50%,2768.000000
75%,2889.000000
max,3955.000000



Technical exclusion bias audit:


,Resistant Excluded,Resistant Retained,Susceptible Excluded,Susceptible Retained,Fisher Odds Ratio,Fisher P Value,Phenotype Used for Cohort Rule
0,2,1601,4,2626,0.820112,1.0,False



Final audit saved to:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/pangenome_final_integrity_audit.csv

PASS: The finalized PLFam pangenome matrix passed the File 05 integrity audit.


# Cell 12 — Finalize File 05 Pangenome Feature Engineering

In [14]:
import os
import numpy as np
import pandas as pd

from scipy import sparse


# =========================================================
# 1. Persistent final-output paths
# =========================================================

X_PANGENOME_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_plfam_binary.npz"
)

PANGENOME_ROW_INDEX_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_row_index.csv"
)

PANGENOME_FEATURE_INDEX_PATH = os.path.join(
    PANGENOME_DIR,
    "X_pangenome_feature_index.csv"
)

PANGENOME_MANIFEST_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_analysis_manifest.csv"
)

PANGENOME_EXCLUDED_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_technical_exclusions.csv"
)

PANGENOME_FINAL_AUDIT_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_final_integrity_audit.csv"
)

PANGENOME_EXCLUSION_BIAS_PATH = os.path.join(
    PANGENOME_DIR,
    "pangenome_exclusion_bias_audit.csv"
)

FILE05_SUMMARY_PATH = os.path.join(
    DRIVE_ROOT,
    "checkpoints",
    "file05_pangenome_feature_engineering_summary.csv"
)


# =========================================================
# 2. Verify required final outputs exist
# =========================================================

required_outputs = [
    X_PANGENOME_PATH,
    PANGENOME_ROW_INDEX_PATH,
    PANGENOME_FEATURE_INDEX_PATH,
    PANGENOME_MANIFEST_PATH,
    PANGENOME_EXCLUDED_PATH,
    PANGENOME_FINAL_AUDIT_PATH,
    PANGENOME_EXCLUSION_BIAS_PATH
]


missing_outputs = [
    path
    for path in required_outputs
    if not os.path.exists(path)
]


assert not missing_outputs, (
    "Missing final File 05 outputs:\n"
    + "\n".join(missing_outputs)
)


# =========================================================
# 3. Reload finalized matrix and indices
# =========================================================

X_pangenome_12 = sparse.load_npz(
    X_PANGENOME_PATH
).tocsr()


df_row_index_12 = pd.read_csv(
    PANGENOME_ROW_INDEX_PATH,
    dtype={
        "Genome ID": str
    }
)


df_feature_index_12 = pd.read_csv(
    PANGENOME_FEATURE_INDEX_PATH,
    dtype={
        "PLFam ID": str
    }
)


df_manifest_12 = pd.read_csv(
    PANGENOME_MANIFEST_PATH,
    dtype={
        "Genome ID": str
    }
)


df_excluded_12 = pd.read_csv(
    PANGENOME_EXCLUDED_PATH,
    dtype={
        "Genome ID": str
    }
)


for df in [
    df_row_index_12,
    df_manifest_12,
    df_excluded_12
]:

    df["Genome ID"] = (
        df["Genome ID"]
        .astype(str)
        .str.strip()
    )


# =========================================================
# 4. Final structural confirmation
# =========================================================

n_genomes, n_features = (
    X_pangenome_12.shape
)


assert (
    n_genomes,
    n_features
) == (
    4227,
    22685
)


assert len(
    df_row_index_12
) == 4227


assert len(
    df_feature_index_12
) == 22685


assert len(
    df_manifest_12
) == 4227


assert len(
    df_excluded_12
) == 6


assert (
    df_row_index_12[
        "Genome ID"
    ]
    .tolist()
    ==
    df_manifest_12[
        "Genome ID"
    ]
    .tolist()
)


# =========================================================
# 5. Binary matrix confirmation
# =========================================================

assert set(
    np.unique(
        X_pangenome_12.data
    )
) == {
    1
}


assert (
    np.asarray(
        X_pangenome_12.sum(
            axis=1
        )
    )
    .ravel()
    > 0
).all()


# =========================================================
# 6. Verify final prevalence rule
# =========================================================

feature_presence = np.asarray(
    X_pangenome_12.sum(
        axis=0
    )
).ravel().astype(int)


feature_absence = (
    n_genomes
    - feature_presence
)


minor_state_count = np.minimum(
    feature_presence,
    feature_absence
)


assert minor_state_count.min() >= 5


# =========================================================
# 7. Final descriptive values
# =========================================================

matrix_nnz = int(
    X_pangenome_12.nnz
)


matrix_density = (
    matrix_nnz
    /
    (
        n_genomes
        * n_features
    )
)


matrix_sparsity = (
    1
    - matrix_density
)


features_per_genome = np.asarray(
    X_pangenome_12.sum(
        axis=1
    )
).ravel().astype(int)


# =========================================================
# 8. Common cohort check against known-AMR matrix
# =========================================================

X_KNOWN_AMR_FINAL_PATH = os.path.join(
    DRIVE_ROOT,
    "data",
    "features",
    "known_amr",
    "X_known_amr_final.csv"
)


df_known_amr_ids = pd.read_csv(
    X_KNOWN_AMR_FINAL_PATH,
    usecols=[
        "Genome ID"
    ],
    dtype={
        "Genome ID": str
    }
)


df_known_amr_ids[
    "Genome ID"
] = (
    df_known_amr_ids[
        "Genome ID"
    ]
    .astype(str)
    .str.strip()
)


known_amr_id_set = set(
    df_known_amr_ids[
        "Genome ID"
    ]
)


pangenome_id_set = set(
    df_row_index_12[
        "Genome ID"
    ]
)


assert pangenome_id_set.issubset(
    known_amr_id_set
)


common_cohort_size = len(
    pangenome_id_set.intersection(
        known_amr_id_set
    )
)


assert common_cohort_size == 4227


# =========================================================
# 9. Create final File 05 summary record
# =========================================================

df_file05_summary = pd.DataFrame(
    [
        {
            "File":
                "05_pangenome_feature_engineering",

            "Status":
                "FINALIZED",

            "Original Study Genomes":
                4233,

            "Pangenome Genomes":
                4227,

            "Technical Exclusions":
                6,

            "Representation":
                "BV-BRC PLFam binary presence/absence",

            "Annotation Source":
                "PATRIC",

            "Minimum PLFam Coverage":
                0.95,

            "Observed PLFams":
                45492,

            "Minor State Threshold":
                5,

            "Final PLFam Features":
                n_features,

            "Matrix Nonzero Entries":
                matrix_nnz,

            "Matrix Density":
                matrix_density,

            "Matrix Sparsity":
                matrix_sparsity,

            "Zero-feature Genomes":
                int(
                    (
                        features_per_genome
                        == 0
                    )
                    .sum()
                ),

            "Common Cohort with Known AMR":
                common_cohort_size,

            "Feature Encoding":
                "Binary presence/absence",

            "Phenotype Used for Cohort Eligibility":
                False,

            "Phenotype Used for Feature Filtering":
                False,

            "Supervised Feature Selection Performed":
                False,

            "Final Matrix":
                X_PANGENOME_PATH,

            "Row Index":
                PANGENOME_ROW_INDEX_PATH,

            "Feature Index":
                PANGENOME_FEATURE_INDEX_PATH
        }
    ]
)


df_file05_summary.to_csv(
    FILE05_SUMMARY_PATH,
    index=False
)


# =========================================================
# 10. Final report
# =========================================================

print("=" * 70)
print("FILE 05 — FINAL STATUS")
print("=" * 70)

print(
    f"Original study genomes     : "
    f"{4_233:,}"
)

print(
    f"Pangenome genomes          : "
    f"{n_genomes:,}"
)

print(
    f"Technical exclusions       : "
    f"{6:,}"
)

print(
    f"PLFam coverage threshold   : "
    f"{95}%"
)

print()

print(
    f"Observed PLFams            : "
    f"{45_492:,}"
)

print(
    f"Minor-state threshold      : "
    f"{5}"
)

print(
    f"Final PLFam features       : "
    f"{n_features:,}"
)

print()

print(
    f"Final matrix               : "
    f"{n_genomes:,} x "
    f"{n_features:,}"
)

print(
    f"Nonzero entries            : "
    f"{matrix_nnz:,}"
)

print(
    f"Matrix density             : "
    f"{matrix_density:.4%}"
)

print(
    f"Matrix sparsity            : "
    f"{matrix_sparsity:.4%}"
)

print(
    f"Zero-feature genomes       : "
    f"{(features_per_genome == 0).sum():,}"
)

print()

print(
    f"Common cohort with X_known_amr: "
    f"{common_cohort_size:,}"
)

print()

print(
    "Phenotype used for cohort rule     : NO"
)

print(
    "Phenotype used for feature filtering: NO"
)

print(
    "Supervised feature selection        : NOT YET"
)

print()

print(
    "Final pangenome matrix:"
)

print(
    X_PANGENOME_PATH
)

print()

print(
    "File 05 summary:"
)

print(
    FILE05_SUMMARY_PATH
)

print()

print(
    "PASS: File 05 pangenome feature engineering "
    "is finalized and locked."
)

FILE 05 — FINAL STATUS
Original study genomes     : 4,233
Pangenome genomes          : 4,227
Technical exclusions       : 6
PLFam coverage threshold   : 95%

Observed PLFams            : 45,492
Minor-state threshold      : 5
Final PLFam features       : 22,685

Final matrix               : 4,227 x 22,685
Nonzero entries            : 11,694,824
Matrix density             : 12.1961%
Matrix sparsity            : 87.8039%
Zero-feature genomes       : 0

Common cohort with X_known_amr: 4,227

Phenotype used for cohort rule     : NO
Phenotype used for feature filtering: NO
Supervised feature selection        : NOT YET

Final pangenome matrix:
/content/drive/MyDrive/AMR-Genome-ML/data/features/pangenome/X_pangenome_plfam_binary.npz

File 05 summary:
/content/drive/MyDrive/AMR-Genome-ML/checkpoints/file05_pangenome_feature_engineering_summary.csv

PASS: File 05 pangenome feature engineering is finalized and locked.
